In [2]:
"""
ipc_hybrid_classify_retrieve_evidence_reason.py
================================================
Full re-implementation matching the "Proposed Hybrid Classify-Retrieve-
Evidence-Reason framework" diagram:

    Case Facts -> [Supervised Classification Branch] + [IPC Retrieval Branch]
               -> Score Normalization and Fusion -> IPC Candidate Ranking
               -> Evidence Sentence Retrieval for each Top-K IPC
               -> LLM-Based Reasoning Generation (Qwen, CoT)
               -> Predicted IPC sections + Evidence Sentences + Explanation

Branch details, exactly as drawn:
  Supervised Classification Branch:
      InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax
      -> 7 IPC probabilities
  IPC Retrieval Branch:
      IPC Knowledge Base -> BM25 + Cosine Similarity -> Retrieval Score
      (for all 575 IPC sections)
  Evidence Sentence Retrieval for Each Top-K IPC:
      Sentence Scoring S = {S1..Sn} via BM25 + Cosine Similarity +
      Classifier-Based Relevance -> Top-m Evidence Sentences
  LLM-Based Reasoning Generation:
      Input to LLM (IPC Section, Selected Evidence Sentences, CoT prompting)
      -> Qwen -> Output (Predicted IPC sections, Evidence Sentences,
      Explanation)

ASSUMPTIONS made explicit (things the diagram doesn't pin down numerically):
  1. The classifier's "Linear & Softmax" head is trained with a SOFT-target
     cross-entropy (multi-hot gold, normalized to sum to 1), not a single
     argmax label -- this lets a doc with 2 gold classifier-class sections
     still supervise the softmax head correctly.
  2. "Retrieval Score (for 575 IPC section)" = normalized BM25 score +
     normalized cosine-similarity score, weighted average (both weights are
     config constants you can tune).
  3. "Score Normalization and Fusion" (top box) = min-max normalize the
     classifier's 7-way distribution and the 575-way retrieval score onto
     the same scale, then weighted-sum them into one fused score per
     section, over the full 575-section space.
  4. Evidence sentence scoring's "Classifier-Based Relevance" term reuses
     the label-wise attention weights (alpha) from the classifier -- i.e.
     how much attention the classifier itself paid to each token/sentence
     for that label -- mapped back onto the case's own sentences. This is
     only available for the 7 classifier-label sections; for retrieval-only
     candidates this term is dropped from the weighted sum (weights
     renormalize automatically).
  5. "Total" in the evaluation table = a simple unweighted mean of the six
     reported metrics (Macro-F1, Micro-F1, Accuracy, ROUGE-L, BLEU, METEOR).
     Change TOTAL_METRIC_WEIGHTS below if your paper defines it differently.
  6. ROUGE-L / BLEU / METEOR score the LLM's generated `explanation` text
     against a reference built by joining that document's gold
     (sentence -> IPC) explanation pairs -- there is no other free-text
     "gold explanation" field in task1.jsonl to compare against.

Expected input files (same as before):
  - task1.jsonl            : one JSON object per line, each with at least
                              {"doc_id": ..., "fact": ..., "statute": [...],
                               "explanation": {sentence: ipc_label, ...}}
  - ipc_sections_clean.json: [{"section": "302", "title": ..., "text": ...}, ...]
"""

# =============================================================================
# STEP 0: DEPENDENCIES
# =============================================================================
import subprocess
import sys


def ensure_packages():
    import importlib
    pkgs = {
        "torch": "torch",
        "transformers": "transformers",
        "scikit-learn": "sklearn",
        "accelerate": "accelerate",
        "rouge_score": "rouge_score",
        "nltk": "nltk",
        "numpy": "numpy",
        "pandas": "pandas",
        "rank_bm25": "rank_bm25",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing '{pip_name}' ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)

    import nltk
    for res, pkg in [("tokenizers/punkt", "punkt"), ("tokenizers/punkt_tab", "punkt_tab"),
                      ("corpora/wordnet", "wordnet"), ("corpora/omw-1.4", "omw-1.4")]:
        try:
            nltk.data.find(res)
        except LookupError:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass


ensure_packages()

import json
import os
import re
import random
import difflib
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer
from rank_bm25 import BM25Okapi
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# =============================================================================
# STEP 1: CONFIG
# =============================================================================
TASK1_PATH = "task1.jsonl"
IPC_KB_PATH = "ipc_sections_clean.json"
OUTPUT_DIR_CLASSIFIER = "./classifier_out"
PREDICTIONS_PATH = "predictions_hybrid.jsonl"
COMPARISON_PATH = "comparison_pred_vs_gold.csv"

RANDOM_SEED = 42
TEST_FRACTION = 0.20
VAL_FRACTION = 0.10

MODEL_NAME = "law-ai/InLegalBERT"

# --- Supervised Classification Branch: InLegalBERT -> BiLSTM -> label-wise
#     Attention -> Linear & Softmax -> 7 IPC probabilities ---
MIN_CLASSIFIER_LABEL_FREQ = 3
CLASSIFIER_MAX_LENGTH = 384
CLASSIFIER_BATCH_SIZE = 8
CLASSIFIER_EPOCHS = 8
CLASSIFIER_LR = 2e-5
BILSTM_HIDDEN = 256          # per direction; BiLSTM output dim = 2 * this
ATTN_DIM = 200               # label-wise attention projection dim

# --- IPC Retrieval Branch: BM25 + Cosine Similarity -> Retrieval Score ---
RETRIEVAL_MAX_TOKEN_LEN = 256
RETRIEVAL_BM25_WEIGHT = 0.5
RETRIEVAL_COSINE_WEIGHT = 0.5

# --- Score Normalization and Fusion -> IPC Candidate Ranking ---
FUSION_CLASSIFIER_WEIGHT = 0.55
FUSION_RETRIEVAL_WEIGHT = 0.45
TOP_K_CANDIDATES = 5          # "Top-K IPCs" fed into evidence retrieval

# --- Evidence Sentence Retrieval for Each Candidate IPC ---
EVIDENCE_BM25_WEIGHT = 0.34
EVIDENCE_COSINE_WEIGHT = 0.33
EVIDENCE_CLS_WEIGHT = 0.33
TOP_M_EVIDENCE = 3            # "Top-m Evidence Sentences" per candidate IPC

# --- Calibration (per-class softmax-probability threshold, on VAL only) ---
THRESHOLD_SEARCH_MIN = 0.02
THRESHOLD_SEARCH_MAX = 0.60
THRESHOLD_SEARCH_STEP = 0.02
DEFAULT_CLASSIFIER_THRESHOLD = 0.15

# --- LLM-Based Reasoning Generation (Qwen, CoT prompting) ---
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # swap for a bigger Qwen if you have the GPU budget
LLM_MAX_NEW_TOKENS = 300
LLM_TEMPERATURE = 0.2

# --- Evaluation ---
TOTAL_METRIC_WEIGHTS = None   # None => simple unweighted mean of all 6 metrics

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# =============================================================================
# STEP 2: SHARED TEXT UTILITIES (sentence splitting + explanation alignment)
# =============================================================================
_ABBREV_PATTERNS = [
    r"\bPW-?\d*\.", r"\bp\.m\.", r"\ba\.m\.", r"\bExt\.-?", r"\bRs\.",
    r"\bNo\.", r"\bSec\.", r"\bSection\.", r"\bvs\.", r"\bv\.", r"\bMr\.",
    r"\bMrs\.", r"\bDr\.", r"\bJ\.\)", r"\bi\.e\.", r"\be\.g\.", r"\bIPC\.",
    r"\bCrPC\.", r"\bHon'ble\.", r"\bU/s\.",
]
_PLACEHOLDER = "<<DOT_{}>>"


def split_sentences_with_spans(text):
    protected = text
    placeholders = {}
    for i, pat in enumerate(_ABBREV_PATTERNS):
        def _sub(m, i=i):
            key = _PLACEHOLDER.format(f"{i}_{len(placeholders)}")
            placeholders[key] = m.group(0)
            return key
        protected = re.sub(pat, _sub, protected)

    raw_sents = re.split(r"(?<=[.!?])\s+(?=[A-Z(\"\u2018\u201c])", protected)

    results = []
    cursor = 0
    for s in raw_sents:
        for key, val in placeholders.items():
            s = s.replace(key, val)
        s_stripped = s.strip()
        if not s_stripped:
            continue
        idx = text.find(s_stripped, cursor)
        if idx == -1:
            idx = text.find(s_stripped)
        if idx == -1:
            start, end = cursor, cursor + len(s_stripped)
        else:
            start, end = idx, idx + len(s_stripped)
        results.append((s_stripped, start, end))
        cursor = end
    return results


def split_sentences(text):
    return [s for s, _, _ in split_sentences_with_spans(text)]


def normalize_ipc_label(label):
    label = str(label).strip()
    m = re.search(r"(\d+[A-Za-z]*)", label)
    if not m:
        return None
    return f"IPC {m.group(1).upper()}"


def align_explanation_to_sentences(fact, explanation, overlap_threshold=0.5):
    sent_spans = split_sentences_with_spans(fact)
    labels = [None] * len(sent_spans)
    for exp_sent, label in explanation.items():
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        start = fact.find(exp_sent)
        if start == -1:
            norm_fact = re.sub(r"\s+", " ", fact)
            n_start = norm_fact.find(exp_norm)
            if n_start != -1:
                start = n_start
        if start == -1:
            sm = difflib.SequenceMatcher(None, fact, exp_norm, autojunk=False)
            match = sm.find_longest_match(0, len(fact), 0, len(exp_norm))
            if match.size < 0.5 * len(exp_norm):
                continue
            start = match.a
            exp_len = len(exp_norm)
        else:
            exp_len = len(exp_sent)
        end = start + exp_len
        for i, (s_text, s_start, s_end) in enumerate(sent_spans):
            overlap = max(0, min(end, s_end) - max(start, s_start))
            if overlap >= overlap_threshold * (s_end - s_start + 1e-6):
                labels[i] = label
    return [(s_text, lab) for (s_text, _, _), lab in zip(sent_spans, labels)]


def load_jsonl_ordered(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_ipc_catalog(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    catalog, titles = {}, {}
    for entry in raw:
        code = str(entry.get("section", "")).strip()
        if not code:
            continue
        title = str(entry.get("title", "")).strip()
        text = str(entry.get("text", "")).strip()
        catalog[code] = f"{title}. {text}" if title else text
        titles[code] = title
    codes = sorted(catalog.keys())
    return catalog, titles, codes


def gold_sections_of(doc):
    return sorted({s for s in (normalize_ipc_label(g) for g in doc.get("statute", [])) if s})


def gold_explanation_reference(doc):
    """Reference text for ROUGE/BLEU/METEOR: gold (sentence -> IPC) pairs
    joined into one paragraph, since that's the only free-text gold
    'reasoning' available in task1.jsonl."""
    exp = doc.get("explanation", {}) or {}
    if not exp:
        return " ".join(split_sentences(doc.get("fact", ""))[:2])
    return " ".join(f"{sent.strip()} (=> {label})" for sent, label in exp.items())


def _catalog_key_for(section_norm, ipc_catalog_raw_keys):
    """'IPC 302' -> the raw catalog key ('302'), used because normalize_ipc_label
    adds the 'IPC ' prefix but ipc_sections_clean.json stores bare section codes."""
    m = re.search(r"(\d+[A-Za-z]*)", section_norm.upper())
    if not m:
        return None
    bare = m.group(1)
    return bare if bare in ipc_catalog_raw_keys else None


# =============================================================================
# STEP 3: LOAD DATA + SPLIT INTO TRAIN / VAL / TEST
# =============================================================================
print("=" * 70)
print("STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits")
print("=" * 70)

ipc_catalog, ipc_titles, all_section_codes_raw = load_ipc_catalog(IPC_KB_PATH)
print(f"Loaded {len(all_section_codes_raw)} official IPC sections from {IPC_KB_PATH}")

# normalized section codes ("IPC 302") mapped back onto raw catalog keys ("302")
norm_to_raw = {normalize_ipc_label(c): c for c in all_section_codes_raw}
all_section_codes = sorted(norm_to_raw.keys())          # "IPC 302", "IPC 498A", ...
section_index = {sec: i for i, sec in enumerate(all_section_codes)}
n_sections = len(all_section_codes)

all_docs = load_jsonl_ordered(TASK1_PATH)
random.Random(RANDOM_SEED).shuffle(all_docs)

n = len(all_docs)
n_test = max(1, int(n * TEST_FRACTION))
n_val = max(1, int((n - n_test) * VAL_FRACTION))
test_docs = all_docs[:n_test]
val_docs = all_docs[n_test:n_test + n_val]
train_docs = all_docs[n_test + n_val:]
print(f"Total docs: {n} | Train: {len(train_docs)} | Val: {len(val_docs)} | Test: {len(test_docs)}")

# =============================================================================
# STEP 4: HYBRID LABEL SPACE (7 classifier classes vs. retrieval-only)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: Building the hybrid label space (classifier vs retrieval-only)")
print("=" * 70)

doc_label_counts = Counter()
for d in train_docs:
    for s in gold_sections_of(d):
        doc_label_counts[s] += 1

classifier_label_list = sorted([s for s, c in doc_label_counts.items() if c >= MIN_CLASSIFIER_LABEL_FREQ])
label_to_idx = {s: i for i, s in enumerate(classifier_label_list)}
idx_to_label = {i: s for s, i in label_to_idx.items()}
num_classifier_labels = len(classifier_label_list)

print(f"{len(doc_label_counts)} distinct gold sections seen in TRAIN docs.")
print(f"-> {num_classifier_labels} kept as CLASSIFIER classes (frequency >= {MIN_CLASSIFIER_LABEL_FREQ}).")
print(f"-> remaining {n_sections - num_classifier_labels} of {n_sections} sections are RETRIEVAL-ONLY.")

# =============================================================================
# STEP 5: SUPERVISED CLASSIFICATION BRANCH
#   InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax
#   -> 7 IPC probabilities
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: Supervised Classification Branch "
      "(InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)")
print("=" * 70)

classifier_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_encoder = AutoModel.from_pretrained(MODEL_NAME)
bert_hidden_size = bert_encoder.config.hidden_size


class LabelWiseAttentionClassifier(nn.Module):
    """InLegalBERT -> BiLSTM -> label-wise attention -> Linear & Softmax.

    For every label c, an attention distribution over the token sequence is
    learned (alpha[:, :, c]); the label's context vector is the attention-
    weighted sum of the BiLSTM outputs; a per-label linear projects that
    context to a single logit; softmax over the `num_labels` logits gives
    the 7 IPC probabilities the diagram asks for. `alpha` is also returned so
    the evidence-retrieval stage can reuse it as "classifier-based
    relevance" per token/sentence, exactly as drawn in the diagram.
    """

    def __init__(self, encoder, hidden_size, num_labels, lstm_hidden=BILSTM_HIDDEN, attn_dim=ATTN_DIM):
        super().__init__()
        self.encoder = encoder
        self.bilstm = nn.LSTM(hidden_size, lstm_hidden, batch_first=True, bidirectional=True)
        lstm_out_dim = lstm_hidden * 2
        self.attn_W = nn.Linear(lstm_out_dim, attn_dim, bias=False)
        self.attn_U = nn.Linear(attn_dim, num_labels, bias=False)
        self.label_weight = nn.Parameter(torch.randn(num_labels, lstm_out_dim) * 0.01)
        self.label_bias = nn.Parameter(torch.zeros(num_labels))
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.bilstm(enc_out)                       # (B, T, 2*lstm_hidden)
        u = torch.tanh(self.attn_W(lstm_out))                     # (B, T, attn_dim)
        scores = self.attn_U(u)                                   # (B, T, num_labels)
        pad_mask = (~attention_mask.bool()).unsqueeze(-1)         # (B, T, 1)
        scores = scores.masked_fill(pad_mask, float("-inf"))
        alpha = torch.softmax(scores, dim=1)                      # per-label attention over tokens
        context = torch.einsum("btl,bth->blh", alpha, lstm_out)   # (B, num_labels, 2*lstm_hidden)
        logits = torch.einsum("blh,lh->bl", context, self.label_weight) + self.label_bias
        probs = torch.softmax(logits, dim=-1)                     # 7 IPC probabilities
        return logits, probs, alpha


classifier_model = LabelWiseAttentionClassifier(bert_encoder, bert_hidden_size, num_classifier_labels).to(DEVICE)


class StatuteDataset(Dataset):
    def __init__(self, records, label_to_idx, tokenizer, max_length):
        self.records = records
        self.label_to_idx = label_to_idx
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(rec["fact"], truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        target = torch.zeros(len(self.label_to_idx))
        for s in gold_sections_of(rec):
            if s in self.label_to_idx:
                target[self.label_to_idx[s]] = 1.0
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": target,
        }


def soft_target_cross_entropy(logits, multi_hot_targets):
    """Cross-entropy against a NORMALIZED multi-hot target (uniform mass over
    every true label present), so the single softmax head can still be
    supervised correctly for documents with >1 gold classifier-class
    section. Falls back to a uniform distribution over all labels for
    documents with none of the 7 classes present (keeps the head well
    defined instead of dividing by zero)."""
    row_sums = multi_hot_targets.sum(dim=-1, keepdim=True)
    safe_targets = torch.where(row_sums > 0, multi_hot_targets / row_sums.clamp(min=1e-9),
                                torch.full_like(multi_hot_targets, 1.0 / multi_hot_targets.size(-1)))
    log_probs = F.log_softmax(logits, dim=-1)
    return -(safe_targets * log_probs).sum(dim=-1).mean()


train_ds = StatuteDataset(train_docs, label_to_idx, classifier_tokenizer, CLASSIFIER_MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=CLASSIFIER_LR)
total_steps = max(1, len(train_loader) * CLASSIFIER_EPOCHS)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                             num_training_steps=total_steps)

classifier_model.train()
for epoch in range(CLASSIFIER_EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits, probs, _ = classifier_model(input_ids, attention_mask)
        loss = soft_target_cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"[classifier] epoch {epoch + 1}/{CLASSIFIER_EPOCHS} -- avg loss: {total_loss / max(1, len(train_loader)):.4f}")

classifier_model.eval()
os.makedirs(OUTPUT_DIR_CLASSIFIER, exist_ok=True)
torch.save(classifier_model.state_dict(), os.path.join(OUTPUT_DIR_CLASSIFIER, "label_attention_classifier.pt"))
classifier_tokenizer.save_pretrained(OUTPUT_DIR_CLASSIFIER)
print(f"Classifier saved to {OUTPUT_DIR_CLASSIFIER}")


@torch.no_grad()
def classifier_forward_for(fact_text):
    """Returns (probs_by_label, alpha, offsets, attn_mask) so downstream steps
    can reuse both the 7 IPC probabilities AND the raw label-wise attention
    for evidence-sentence relevance."""
    enc = classifier_tokenizer(fact_text, truncation=True, padding="max_length",
                                max_length=CLASSIFIER_MAX_LENGTH, return_tensors="pt",
                                return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    logits, probs, alpha = classifier_model(enc["input_ids"], enc["attention_mask"])
    probs = probs.squeeze(0).cpu().numpy()
    alpha = alpha.squeeze(0).cpu().numpy()          # (T, num_labels)
    attn_mask = enc["attention_mask"].squeeze(0).cpu().numpy()
    probs_by_label = {idx_to_label[i]: float(probs[i]) for i in range(num_classifier_labels)}
    return probs_by_label, alpha, offsets.numpy(), attn_mask


def classifier_attention_per_sentence(fact_text, label, alpha, offsets, attn_mask):
    """Aggregates the label-wise attention mass (alpha[:, label_idx]) that
    fell on each sentence's character span -- this IS the 'Classifier-Based
    Relevance' term drawn in the Evidence Sentence Retrieval box."""
    if label not in label_to_idx:
        return None
    lab_idx = label_to_idx[label]
    sent_spans = split_sentences_with_spans(fact_text)
    if not sent_spans:
        return []
    scores = [0.0] * len(sent_spans)
    for t in range(len(offsets)):
        if attn_mask[t] == 0:
            continue
        tok_start, tok_end = int(offsets[t][0]), int(offsets[t][1])
        if tok_end <= tok_start:
            continue
        for si, (_, s_start, s_end) in enumerate(sent_spans):
            if tok_start < s_end and tok_end > s_start:
                scores[si] += float(alpha[t, lab_idx])
                break
    return scores

# =============================================================================
# STEP 6: IPC RETRIEVAL BRANCH
#   IPC Knowledge Base -> BM25 + Cosine Similarity -> Retrieval Score
#   (for 575 IPC sections)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)")
print("=" * 70)

retrieval_tokenizer = classifier_tokenizer  # frozen InLegalBERT for embeddings, no separate fine-tuning stage
retrieval_encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()


def _bm25_tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


print("Building BM25 index over the IPC Knowledge Base ...")
catalog_texts = [ipc_catalog[norm_to_raw[sec]] for sec in all_section_codes]
bm25_corpus_tokens = [_bm25_tokenize(t) for t in catalog_texts]
bm25_index = BM25Okapi(bm25_corpus_tokens)


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def encode_texts(texts, max_len, batch_size=32):
    embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = retrieval_tokenizer(batch, truncation=True, padding=True, max_length=max_len,
                                   return_tensors="pt").to(DEVICE)
        out = retrieval_encoder(**enc).last_hidden_state
        pooled = F.normalize(mean_pool(out, enc["attention_mask"]), p=2, dim=-1)
        embeds.append(pooled.cpu())
    return torch.cat(embeds, dim=0) if embeds else torch.zeros((0, bert_hidden_size))


print("Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...")
catalog_embeddings = encode_texts(catalog_texts, RETRIEVAL_MAX_TOKEN_LEN)
print(f"Catalog embedding matrix: {tuple(catalog_embeddings.shape)}")


def _min_max_norm(arr):
    arr = np.asarray(arr, dtype=np.float64)
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-9:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)


def retrieval_scores_for(fact_text):
    """BM25 + Cosine Similarity -> one fused Retrieval Score per of the 575
    IPC sections, plus the best-supporting sentence per section (used later
    for evidence retrieval / exact_fact fallback)."""
    sentences = split_sentences(fact_text)
    if not sentences:
        return {sec: 0.0 for sec in all_section_codes}, {}

    bm25_query_tokens = _bm25_tokenize(fact_text)
    bm25_scores = bm25_index.get_scores(bm25_query_tokens)           # (n_sections,)

    sent_embeddings = encode_texts(sentences, RETRIEVAL_MAX_TOKEN_LEN)
    sims = (sent_embeddings @ catalog_embeddings.t()).numpy()        # (n_sent, n_sections)
    best_idx_per_section = sims.argmax(axis=0)
    cosine_scores = sims.max(axis=0)

    bm25_norm = _min_max_norm(bm25_scores)
    cosine_norm = _min_max_norm(cosine_scores)
    fused = RETRIEVAL_BM25_WEIGHT * bm25_norm + RETRIEVAL_COSINE_WEIGHT * cosine_norm

    scores = {all_section_codes[j]: float(fused[j]) for j in range(n_sections)}
    best_sentence = {all_section_codes[j]: sentences[best_idx_per_section[j]] for j in range(n_sections)}
    return scores, best_sentence

# =============================================================================
# STEP 7: CALIBRATION ON THE VALIDATION SET
#   Per-classifier-class threshold on the softmax probability (not sigmoid),
#   searched to maximize each class's own val F1.
# =============================================================================
print("\n" + "=" * 70)
print("STEP 7: Calibrating per-class classifier thresholds on VAL")
print("=" * 70)

val_gold = {d["doc_id"]: gold_sections_of(d) for d in val_docs}
val_probs_matrix = np.zeros((len(val_docs), num_classifier_labels))
val_alpha_cache, val_offsets_cache, val_mask_cache = {}, {}, {}
for i, d in enumerate(val_docs):
    probs, alpha, offsets, mask = classifier_forward_for(d["fact"])
    for j, lab in enumerate(classifier_label_list):
        val_probs_matrix[i, j] = probs[lab]

val_targets_matrix = np.zeros_like(val_probs_matrix)
for i, d in enumerate(val_docs):
    gold = set(val_gold[d["doc_id"]])
    for j, lab in enumerate(classifier_label_list):
        val_targets_matrix[i, j] = 1.0 if lab in gold else 0.0

classifier_thresholds = {}
grid = np.arange(THRESHOLD_SEARCH_MIN, THRESHOLD_SEARCH_MAX + 1e-9, THRESHOLD_SEARCH_STEP)
for j, lab in enumerate(classifier_label_list):
    y_true = val_targets_matrix[:, j]
    if y_true.sum() == 0:
        classifier_thresholds[lab] = DEFAULT_CLASSIFIER_THRESHOLD
        continue
    best_t, best_f1 = DEFAULT_CLASSIFIER_THRESHOLD, -1.0
    for t in grid:
        y_pred = (val_probs_matrix[:, j] >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    classifier_thresholds[lab] = best_t
print(f"Calibrated {len(classifier_thresholds)} per-class thresholds "
      f"(mean={np.mean(list(classifier_thresholds.values())):.3f}).")

# =============================================================================
# STEP 8: SCORE NORMALIZATION AND FUSION -> IPC CANDIDATE RANKING
# =============================================================================
print("\n" + "=" * 70)
print("STEP 8: Score Normalization and Fusion -> IPC Candidate Ranking")
print("=" * 70)


def fused_candidate_ranking(fact_text):
    """Combines the Supervised Classification Branch (7-way softmax) with the
    IPC Retrieval Branch (575-way BM25+cosine score) into one ranked list
    over the FULL 575-section space, exactly matching the 'Score
    Normalization and Fusion -> IPC Candidate Ranking' boxes."""
    cls_probs, alpha, offsets, mask = classifier_forward_for(fact_text)
    retr_scores, retr_best_sentence = retrieval_scores_for(fact_text)

    cls_vector = np.zeros(n_sections)
    for lab, p in cls_probs.items():
        cls_vector[section_index[lab]] = p
    retr_vector = np.array([retr_scores[sec] for sec in all_section_codes])

    cls_norm = _min_max_norm(cls_vector)
    retr_norm = _min_max_norm(retr_vector)
    fused = FUSION_CLASSIFIER_WEIGHT * cls_norm + FUSION_RETRIEVAL_WEIGHT * retr_norm

    ranking = sorted(range(n_sections), key=lambda i: -fused[i])
    candidates = []
    for i in ranking[:TOP_K_CANDIDATES]:
        sec = all_section_codes[i]
        source = "classifier" if (sec in label_to_idx and cls_probs[sec] >= classifier_thresholds[sec]) else "retrieval"
        candidates.append({
            "section": sec,
            "fused_score": float(fused[i]),
            "classifier_score": float(cls_vector[i]),
            "retrieval_score": float(retr_vector[i]),
            "source": source,
        })
    return candidates, cls_probs, alpha, offsets, mask, retr_best_sentence

# =============================================================================
# STEP 9: EVIDENCE SENTENCE RETRIEVAL FOR EACH CANDIDATE IPC
#   Sentence Scoring S = {S1..Sn} via BM25 + Cosine Similarity +
#   Classifier-Based Relevance -> Top-m Evidence Sentences
# =============================================================================
print("\n" + "=" * 70)
print("STEP 9: Evidence Sentence Retrieval for each Top-K candidate IPC")
print("=" * 70)


def evidence_sentences_for_candidate(fact_text, section, alpha, offsets, mask):
    sentences = split_sentences(fact_text)
    if not sentences:
        return []
    section_text = ipc_catalog[norm_to_raw[section]]

    # -- BM25 over the case's own sentences, queried with the section text --
    sent_tokens = [_bm25_tokenize(s) for s in sentences]
    local_bm25 = BM25Okapi(sent_tokens) if sent_tokens else None
    bm25_scores = local_bm25.get_scores(_bm25_tokenize(section_text)) if local_bm25 else np.zeros(len(sentences))

    # -- Cosine similarity: each sentence vs the section text --
    sent_embs = encode_texts(sentences, RETRIEVAL_MAX_TOKEN_LEN)
    section_emb = encode_texts([section_text], RETRIEVAL_MAX_TOKEN_LEN)
    cosine_scores = (sent_embs @ section_emb.t()).squeeze(-1).numpy()

    # -- Classifier-based relevance: label-wise attention mass per sentence --
    cls_scores = classifier_attention_per_sentence(fact_text, section, alpha, offsets, mask)

    bm25_norm = _min_max_norm(bm25_scores)
    cosine_norm = _min_max_norm(cosine_scores)
    if cls_scores is not None:
        cls_norm = _min_max_norm(cls_scores)
        combined = (EVIDENCE_BM25_WEIGHT * bm25_norm + EVIDENCE_COSINE_WEIGHT * cosine_norm
                    + EVIDENCE_CLS_WEIGHT * cls_norm)
    else:
        # renormalize the two remaining weights when this section has no
        # classifier attention available (retrieval-only section)
        w_sum = EVIDENCE_BM25_WEIGHT + EVIDENCE_COSINE_WEIGHT
        combined = (EVIDENCE_BM25_WEIGHT / w_sum) * bm25_norm + (EVIDENCE_COSINE_WEIGHT / w_sum) * cosine_norm

    top_idx = np.argsort(-combined)[:TOP_M_EVIDENCE]
    return [sentences[i] for i in sorted(top_idx)]   # keep original document order

# =============================================================================
# STEP 10: LLM-BASED REASONING GENERATION (Qwen, Chain-of-Thought prompting)
#   Input to LLM (IPC Section, Selected Evidence Sentences, CoT Prompting)
#   -> Qwen -> Output (Predicted IPC sections, Evidence Sentences, Explanation)
# =============================================================================
print("\n" + "=" * 70)
print(f"STEP 10: Loading Qwen ({QWEN_MODEL_NAME}) for reasoning generation")
print("=" * 70)

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE).eval()


def _build_cot_prompt(fact_snippet, section, section_title, evidence_sentences):
    evidence_block = "\n".join(f"- {s}" for s in evidence_sentences) or "(no distinct evidence sentence found)"
    return (
        f"You are a legal reasoning assistant for Indian Penal Code (IPC) section attribution.\n\n"
        f"Case fact (relevant excerpt):\n{fact_snippet}\n\n"
        f"Candidate IPC section: {section} ({section_title})\n"
        f"Evidence sentences retrieved from the case for this section:\n{evidence_block}\n\n"
        f"Think step by step (chain of thought): first restate what {section} legally requires, "
        f"then check whether the evidence sentences above satisfy each requirement, then decide.\n"
        f"Respond with ONLY a JSON object, no extra text, in this exact schema:\n"
        f'{{"applies": true or false, "evidence_sentences": [the evidence sentences you actually relied on], '
        f'"explanation": "one or two sentence justification"}}'
    )


@torch.no_grad()
def qwen_reason_about_candidate(fact_text, section, evidence_sentences):
    section_title = ipc_titles.get(section, "")
    fact_snippet = fact_text[:1500]
    prompt = _build_cot_prompt(fact_snippet, section, section_title, evidence_sentences)
    messages = [{"role": "user", "content": prompt}]
    # NOTE: newer `transformers` returns a BatchEncoding (dict-like: input_ids +
    # attention_mask) from apply_chat_template when return_tensors="pt", NOT a
    # bare tensor -- `.to(DEVICE)` on a BatchEncoding returns the same
    # dict-like object, so indexing it with `.shape` (as if it were a tensor)
    # fails with the KeyError/AttributeError you hit. Fix: ask for the dict
    # explicitly and pull input_ids / attention_mask out of it by name.
    encoded = qwen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(DEVICE)
    input_ids = encoded["input_ids"]
    attention_mask = encoded.get("attention_mask")
    output_ids = qwen_model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        max_new_tokens=LLM_MAX_NEW_TOKENS, do_sample=LLM_TEMPERATURE > 0,
        temperature=max(LLM_TEMPERATURE, 1e-5), pad_token_id=qwen_tokenizer.eos_token_id,
    )
    generated = qwen_tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)

    parsed = {"applies": True, "evidence_sentences": evidence_sentences, "explanation": generated.strip()}
    match = re.search(r"\{.*\}", generated, flags=re.DOTALL)
    if match:
        try:
            candidate = json.loads(match.group(0))
            parsed["applies"] = bool(candidate.get("applies", True))
            parsed["evidence_sentences"] = candidate.get("evidence_sentences", evidence_sentences) or evidence_sentences
            parsed["explanation"] = str(candidate.get("explanation", "")).strip() or generated.strip()
        except Exception:
            pass
    return parsed


def predict_document_hybrid(fact_text):
    """Runs the full pipeline for one case: Candidate Ranking -> Evidence
    Retrieval -> LLM Reasoning -> final predicted sections/evidence/explanation."""
    candidates, cls_probs, alpha, offsets, mask, retr_best_sentence = fused_candidate_ranking(fact_text)

    results = []
    for cand in candidates:
        section = cand["section"]
        evidence = evidence_sentences_for_candidate(fact_text, section, alpha, offsets, mask)
        if not evidence:
            fallback = retr_best_sentence.get(section, "")
            evidence = [fallback] if fallback else []
        llm_out = qwen_reason_about_candidate(fact_text, section, evidence)
        results.append({
            "section": section,
            "fused_score": cand["fused_score"],
            "source": cand["source"],
            "evidence_sentences": llm_out["evidence_sentences"],
            "explanation": llm_out["explanation"],
            "applies": llm_out["applies"],
        })

    predicted = [r for r in results if r["applies"]]
    if not predicted:
        predicted = results[:1]   # never emit nothing: keep the single best-ranked candidate
    return predicted

# =============================================================================
# STEP 11: RUN ON THE HELD-OUT TEST SET
# =============================================================================
print("\n" + "=" * 70)
print(f"STEP 11: Predicting on {len(test_docs)} held-out TEST documents")
print("=" * 70)

all_predictions = {}
for d in test_docs:
    statute_preds = predict_document_hybrid(d["fact"])
    all_predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": statute_preds}
    pred_secs = [p["section"] for p in statute_preds]
    print(f"{d['doc_id']} -> pred={pred_secs} | gold={gold_sections_of(d)}")

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for rec in all_predictions.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"\nPredictions saved to: {PREDICTIONS_PATH}")

# =============================================================================
# STEP 12: EVALUATION -- Macro-F1 / Micro-F1 / Accuracy (classification) +
#          ROUGE-L / BLEU / METEOR (generated explanation quality) + Total
# =============================================================================
print("\n" + "=" * 70)
print("STEP 12: Evaluation against gold labels (Table-4 style report)")
print("=" * 70)


def pred_sections_of(pred_rec):
    return [p["section"] for p in pred_rec.get("statute", [])]


def pred_explanation_of(pred_rec):
    return " ".join(p.get("explanation", "") for p in pred_rec.get("statute", []))


rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smoothing = SmoothingFunction().method1


def compute_classification_metrics(test_docs, predictions):
    gold_labels = [gold_sections_of(d) for d in test_docs]
    pred_labels = [pred_sections_of(predictions[d["doc_id"]]) for d in test_docs]
    all_labels = sorted(set(l for labels in (gold_labels + pred_labels) for l in labels))
    mlb = MultiLabelBinarizer(classes=all_labels)
    y_true = mlb.fit_transform(gold_labels)
    y_pred = mlb.transform(pred_labels)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    exact_matches = sum(
        1 for d in test_docs
        if set(pred_sections_of(predictions[d["doc_id"]])) == set(gold_sections_of(d))
    )
    accuracy = exact_matches / len(test_docs)
    return macro_f1, micro_f1, accuracy


def compute_generation_metrics(test_docs, predictions):
    rouge_l_scores, bleu_scores, meteor_scores = [], [], []
    for d in test_docs:
        reference = gold_explanation_reference(d)
        hypothesis = pred_explanation_of(predictions[d["doc_id"]])
        if not hypothesis.strip():
            continue
        rouge_l_scores.append(rouge.score(reference, hypothesis)["rougeL"].fmeasure)
        ref_tokens = reference.split()
        hyp_tokens = hypothesis.split()
        bleu_scores.append(sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing))
        try:
            meteor_scores.append(meteor_score([ref_tokens], hyp_tokens))
        except Exception:
            pass
    rouge_l = float(np.mean(rouge_l_scores)) if rouge_l_scores else 0.0
    bleu = float(np.mean(bleu_scores)) if bleu_scores else 0.0
    meteor = float(np.mean(meteor_scores)) if meteor_scores else 0.0
    return rouge_l, bleu, meteor


macro_f1, micro_f1, accuracy = compute_classification_metrics(test_docs, all_predictions)
rouge_l, bleu, meteor = compute_generation_metrics(test_docs, all_predictions)

metric_values = {
    "Macro-F1": macro_f1, "Micro-F1": micro_f1, "Accuracy": accuracy,
    "ROUGE-L": rouge_l, "BLEU": bleu, "METEOR": meteor,
}
if TOTAL_METRIC_WEIGHTS is None:
    total = float(np.mean(list(metric_values.values())))
else:
    total = float(sum(metric_values[k] * TOTAL_METRIC_WEIGHTS[k] for k in metric_values))

print("\nTable 4-style Performance report")
header = "Run       " + "  ".join(f"{k:>10s}" for k in metric_values) + f"  {'Total':>10s}"
print(header)
row = "Run 1     " + "  ".join(f"{metric_values[k]:>10.4f}" for k in metric_values) + f"  {total:>10.4f}"
print(row)

# =============================================================================
# STEP 13: SIDE-BY-SIDE COMPARISON CSV
# =============================================================================
print("\n" + "=" * 70)
print("STEP 13: Writing predicted-vs-gold comparison CSV")
print("=" * 70)


def write_comparison_csv(test_docs, predictions, path):
    import csv
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "doc_id", "fact_snippet", "gold_sections", "predicted_sections",
            "correct_sections", "missed_sections", "extra_sections", "exact_match",
            "predicted_explanation",
        ])
        for d in test_docs:
            gold = set(gold_sections_of(d))
            pred = set(pred_sections_of(predictions[d["doc_id"]]))
            correct = sorted(gold & pred)
            missed = sorted(gold - pred)
            extra = sorted(pred - gold)
            fact_snippet = (d.get("fact", "") or "")[:150].replace("\n", " ")
            writer.writerow([
                d["doc_id"], fact_snippet,
                "; ".join(sorted(gold)), "; ".join(sorted(pred)),
                "; ".join(correct), "; ".join(missed), "; ".join(extra),
                "YES" if gold == pred else "NO",
                pred_explanation_of(predictions[d["doc_id"]])[:300],
            ])
    print(f"Wrote predicted-vs-gold comparison to: {path}")


write_comparison_csv(test_docs, all_predictions, COMPARISON_PATH)
print("\nDONE.")

Torch: 2.14.0+cu130 | CUDA available: True
Device: cuda
STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits
Loaded 575 official IPC sections from ipc_sections_clean.json
Total docs: 525 | Train: 378 | Val: 42 | Test: 105

STEP 4: Building the hybrid label space (classifier vs retrieval-only)
7 distinct gold sections seen in TRAIN docs.
-> 7 kept as CLASSIFIER classes (frequency >= 3).
-> remaining 568 of 575 sections are RETRIEVAL-ONLY.

STEP 5: Supervised Classification Branch (InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[classifier] epoch 1/8 -- avg loss: 1.9166
[classifier] epoch 2/8 -- avg loss: 1.6166
[classifier] epoch 3/8 -- avg loss: 1.2710
[classifier] epoch 4/8 -- avg loss: 1.0573
[classifier] epoch 5/8 -- avg loss: 0.8968
[classifier] epoch 6/8 -- avg loss: 0.7957
[classifier] epoch 7/8 -- avg loss: 0.7258
[classifier] epoch 8/8 -- avg loss: 0.6944
Classifier saved to ./classifier_out

STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building BM25 index over the IPC Knowledge Base ...
Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...
Catalog embedding matrix: (575, 768)

STEP 7: Calibrating per-class classifier thresholds on VAL
Calibrated 7 per-class thresholds (mean=0.217).

STEP 8: Score Normalization and Fusion -> IPC Candidate Ranking

STEP 9: Evidence Sentence Retrieval for each Top-K candidate IPC

STEP 10: Loading Qwen (Qwen/Qwen2.5-1.5B-Instruct) for reasoning generation


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


STEP 11: Predicting on 105 held-out TEST documents
2002.INSC.274.txt -> pred=['IPC 376'] | gold=['IPC 376']
1999.INSC.378.txt -> pred=['IPC 376', 'IPC 354D'] | gold=['IPC 201', 'IPC 302']
2012.INSC.512.txt -> pred=['IPC 302'] | gold=['IPC 498A']
1998.INSC.126.txt -> pred=['IPC 302'] | gold=['IPC 302']
2007.INSC.590.txt -> pred=['IPC 302'] | gold=['IPC 147']
2013.INSC.960.txt -> pred=['IPC 420'] | gold=['IPC 201']
2009.INSC.1130.txt -> pred=['IPC 376', 'IPC 376AB'] | gold=['IPC 376']
1999.INSC.175.txt -> pred=['IPC 302'] | gold=['IPC 302']
2003.INSC.597.txt -> pred=['IPC 302'] | gold=['IPC 302']
1998.INSC.474.txt -> pred=['IPC 302'] | gold=['IPC 302']
2015.INSC.345.txt -> pred=['IPC 147'] | gold=['IPC 302', 'IPC 506']
2011.INSC.316.txt -> pred=['IPC 147', 'IPC 154'] | gold=['IPC 147', 'IPC 302']
2016.INSC.429.txt -> pred=['IPC 354C'] | gold=['IPC 201']
2007.INSC.291.txt -> pred=['IPC 506'] | gold=['IPC 147', 'IPC 302', 'IPC 506']
1996.INSC.1547.txt -> pred=['IPC 302'] | gold=['IPC 302'

In [3]:
"""
ipc_hybrid_statute_only.py
==========================
Simplified / fast version of the Classify-Retrieve-Evidence-Reason pipeline:
keeps ONLY the two branches that decide which statutes apply, drops the
Evidence Sentence Retrieval box and the LLM-Based Reasoning box entirely.

    Case Facts -> [Supervised Classification Branch] + [IPC Retrieval Branch]
               -> Score Normalization and Fusion
               -> THRESHOLDED statute prediction (multi-label decision)
               -> Predicted IPC sections

WHY THE PREVIOUS RUN SCORED MACRO-F1 = 0.14 (root cause + fix)
----------------------------------------------------------------
The previous `predict_document_hybrid` always emitted the top-TOP_K_CANDIDATES
(5) fused candidates as the FINAL prediction for every document (the LLM
reasoning stage was supposed to filter those 5 down, but with reasoning
removed nothing filtered them anymore). Since 568 of 575 IPC sections are
retrieval-only "noise" sections that almost never are actually correct, 3-4
of those 5 candidates per document were usually wrong. Each wrong section
becomes its OWN class with F1 = 0 in a macro-average, so ~30+ distinct
spuriously-predicted classes dragged Macro-F1 down to 0.14 even though
Micro-F1 (dominated by the few frequent, mostly-correct classes) still
looked fine at 0.53.

THE FIX in this version:
  1. The classifier's calibrated per-class thresholds (Step 7) are now
     ACTUALLY USED to decide the predicted set: a section from the 7
     classifier classes is predicted iff its softmax probability clears its
     own calibrated threshold -- true multi-label thresholding, not "always
     top-K".
  2. A retrieval-only section (one of the other 568) is only added to the
     prediction if its fused retrieval score clears a conservative fixed
     bar (RETRIEVAL_ONLY_THRESHOLD, default 0.85) -- this is what stops the
     old spam-prediction bug from reappearing. Since retrieval-only
     sections rarely have reliable supervision signal here, this is a
     high, deliberately conservative bar rather than something grid-searched
     to a degenerate low value.
  3. If nothing clears any threshold, we fall back to the single best fused
     candidate (never emit an empty prediction) -- this is what keeps
     Accuracy/Micro-F1 sane for docs sitting right at the decision boundary.

Everything else (BiLSTM + label-wise attention classifier, BM25 + cosine
retrieval, min-max fusion) is UNCHANGED from before -- same architecture,
just a corrected decision rule and the evidence/LLM stages removed for
speed.

Expected input files (unchanged):
  - task1.jsonl            : one JSON object per line, each with at least
                              {"doc_id": ..., "fact": ..., "statute": [...]}
  - ipc_sections_clean.json: [{"section": "302", "title": ..., "text": ...}, ...]
"""

# =============================================================================
# STEP 0: DEPENDENCIES
# =============================================================================
import subprocess
import sys


def ensure_packages():
    import importlib
    pkgs = {
        "torch": "torch",
        "transformers": "transformers",
        "scikit-learn": "sklearn",
        "numpy": "numpy",
        "rank_bm25": "rank_bm25",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing '{pip_name}' ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import json
import os
import re
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from rank_bm25 import BM25Okapi

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# =============================================================================
# STEP 1: CONFIG
# =============================================================================
TASK1_PATH = "task1.jsonl"
IPC_KB_PATH = "ipc_sections_clean.json"
OUTPUT_DIR_CLASSIFIER = "./classifier_out"
PREDICTIONS_PATH = "predictions_statute_only.jsonl"
COMPARISON_PATH = "comparison_pred_vs_gold.csv"

RANDOM_SEED = 42
TEST_FRACTION = 0.20
VAL_FRACTION = 0.10

MODEL_NAME = "law-ai/InLegalBERT"

# --- Supervised Classification Branch: InLegalBERT -> BiLSTM -> label-wise
#     Attention -> Linear & Softmax -> 7 IPC probabilities ---
MIN_CLASSIFIER_LABEL_FREQ = 3
CLASSIFIER_MAX_LENGTH = 384
CLASSIFIER_BATCH_SIZE = 8
CLASSIFIER_EPOCHS = 8
CLASSIFIER_LR = 2e-5
BILSTM_HIDDEN = 256
ATTN_DIM = 200

# --- IPC Retrieval Branch: BM25 + Cosine Similarity -> Retrieval Score ---
RETRIEVAL_MAX_TOKEN_LEN = 256
RETRIEVAL_BM25_WEIGHT = 0.5
RETRIEVAL_COSINE_WEIGHT = 0.5

# --- Score Normalization and Fusion (used only for the top-1 fallback now) ---
FUSION_CLASSIFIER_WEIGHT = 0.55
FUSION_RETRIEVAL_WEIGHT = 0.45

# --- Calibration: per-class softmax-probability threshold, searched on VAL
#     to maximize each class's own F1 -- THIS is what actually gates the
#     final prediction now (previously computed but unused). ---
THRESHOLD_SEARCH_MIN = 0.02
THRESHOLD_SEARCH_MAX = 0.60
THRESHOLD_SEARCH_STEP = 0.02
DEFAULT_CLASSIFIER_THRESHOLD = 0.15

# --- Retrieval-only sections (568 of 575) get a single conservative fixed
#     bar instead of being grid-searched to a degenerate low value -- this
#     is precisely the guard that fixes the old spam-prediction bug. Set to
#     1.01 (unreachable) to disable retrieval-only predictions altogether. ---
ENABLE_RETRIEVAL_ONLY_PREDICTIONS = True
RETRIEVAL_ONLY_THRESHOLD = 0.85

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# =============================================================================
# STEP 2: SHARED TEXT UTILITIES
# =============================================================================
def normalize_ipc_label(label):
    label = str(label).strip()
    m = re.search(r"(\d+[A-Za-z]*)", label)
    if not m:
        return None
    return f"IPC {m.group(1).upper()}"


def load_jsonl_ordered(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_ipc_catalog(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    catalog = {}
    for entry in raw:
        code = str(entry.get("section", "")).strip()
        if not code:
            continue
        title = str(entry.get("title", "")).strip()
        text = str(entry.get("text", "")).strip()
        catalog[code] = f"{title}. {text}" if title else text
    return catalog, sorted(catalog.keys())


def gold_sections_of(doc):
    return sorted({s for s in (normalize_ipc_label(g) for g in doc.get("statute", [])) if s})


def _bm25_tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def _min_max_norm(arr):
    arr = np.asarray(arr, dtype=np.float64)
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-9:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

# =============================================================================
# STEP 3: LOAD DATA + SPLIT INTO TRAIN / VAL / TEST
# =============================================================================
print("=" * 70)
print("STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits")
print("=" * 70)

ipc_catalog, all_section_codes_raw = load_ipc_catalog(IPC_KB_PATH)
print(f"Loaded {len(all_section_codes_raw)} official IPC sections from {IPC_KB_PATH}")

norm_to_raw = {normalize_ipc_label(c): c for c in all_section_codes_raw}
all_section_codes = sorted(norm_to_raw.keys())
section_index = {sec: i for i, sec in enumerate(all_section_codes)}
n_sections = len(all_section_codes)

all_docs = load_jsonl_ordered(TASK1_PATH)
random.Random(RANDOM_SEED).shuffle(all_docs)

n = len(all_docs)
n_test = max(1, int(n * TEST_FRACTION))
n_val = max(1, int((n - n_test) * VAL_FRACTION))
test_docs = all_docs[:n_test]
val_docs = all_docs[n_test:n_test + n_val]
train_docs = all_docs[n_test + n_val:]
print(f"Total docs: {n} | Train: {len(train_docs)} | Val: {len(val_docs)} | Test: {len(test_docs)}")

# =============================================================================
# STEP 4: HYBRID LABEL SPACE (7 classifier classes vs retrieval-only)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: Building the hybrid label space (classifier vs retrieval-only)")
print("=" * 70)

doc_label_counts = Counter()
for d in train_docs:
    for s in gold_sections_of(d):
        doc_label_counts[s] += 1

classifier_label_list = sorted([s for s, c in doc_label_counts.items() if c >= MIN_CLASSIFIER_LABEL_FREQ])
label_to_idx = {s: i for i, s in enumerate(classifier_label_list)}
idx_to_label = {i: s for s, i in label_to_idx.items()}
num_classifier_labels = len(classifier_label_list)

print(f"{len(doc_label_counts)} distinct gold sections seen in TRAIN docs.")
print(f"-> {num_classifier_labels} kept as CLASSIFIER classes (frequency >= {MIN_CLASSIFIER_LABEL_FREQ}).")
print(f"-> remaining {n_sections - num_classifier_labels} of {n_sections} sections are RETRIEVAL-ONLY.")

# =============================================================================
# STEP 5: SUPERVISED CLASSIFICATION BRANCH
#   InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax
#   -> 7 IPC probabilities
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: Supervised Classification Branch "
      "(InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)")
print("=" * 70)

classifier_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_encoder = AutoModel.from_pretrained(MODEL_NAME)
bert_hidden_size = bert_encoder.config.hidden_size


class LabelWiseAttentionClassifier(nn.Module):
    """InLegalBERT -> BiLSTM -> label-wise attention -> Linear & Softmax."""

    def __init__(self, encoder, hidden_size, num_labels, lstm_hidden=BILSTM_HIDDEN, attn_dim=ATTN_DIM):
        super().__init__()
        self.encoder = encoder
        self.bilstm = nn.LSTM(hidden_size, lstm_hidden, batch_first=True, bidirectional=True)
        lstm_out_dim = lstm_hidden * 2
        self.attn_W = nn.Linear(lstm_out_dim, attn_dim, bias=False)
        self.attn_U = nn.Linear(attn_dim, num_labels, bias=False)
        self.label_weight = nn.Parameter(torch.randn(num_labels, lstm_out_dim) * 0.01)
        self.label_bias = nn.Parameter(torch.zeros(num_labels))
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.bilstm(enc_out)
        u = torch.tanh(self.attn_W(lstm_out))
        scores = self.attn_U(u)
        pad_mask = (~attention_mask.bool()).unsqueeze(-1)
        scores = scores.masked_fill(pad_mask, float("-inf"))
        alpha = torch.softmax(scores, dim=1)
        context = torch.einsum("btl,bth->blh", alpha, lstm_out)
        logits = torch.einsum("blh,lh->bl", context, self.label_weight) + self.label_bias
        probs = torch.softmax(logits, dim=-1)
        return logits, probs


classifier_model = LabelWiseAttentionClassifier(bert_encoder, bert_hidden_size, num_classifier_labels).to(DEVICE)


class StatuteDataset(Dataset):
    def __init__(self, records, label_to_idx, tokenizer, max_length):
        self.records = records
        self.label_to_idx = label_to_idx
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(rec["fact"], truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        target = torch.zeros(len(self.label_to_idx))
        for s in gold_sections_of(rec):
            if s in self.label_to_idx:
                target[self.label_to_idx[s]] = 1.0
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": target,
        }


def soft_target_cross_entropy(logits, multi_hot_targets):
    """Cross-entropy against a NORMALIZED multi-hot target so the single
    softmax head is still correctly supervised for multi-label documents."""
    row_sums = multi_hot_targets.sum(dim=-1, keepdim=True)
    safe_targets = torch.where(row_sums > 0, multi_hot_targets / row_sums.clamp(min=1e-9),
                                torch.full_like(multi_hot_targets, 1.0 / multi_hot_targets.size(-1)))
    log_probs = F.log_softmax(logits, dim=-1)
    return -(safe_targets * log_probs).sum(dim=-1).mean()


train_ds = StatuteDataset(train_docs, label_to_idx, classifier_tokenizer, CLASSIFIER_MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=CLASSIFIER_LR)
total_steps = max(1, len(train_loader) * CLASSIFIER_EPOCHS)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                             num_training_steps=total_steps)

classifier_model.train()
for epoch in range(CLASSIFIER_EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits, probs = classifier_model(input_ids, attention_mask)
        loss = soft_target_cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"[classifier] epoch {epoch + 1}/{CLASSIFIER_EPOCHS} -- avg loss: {total_loss / max(1, len(train_loader)):.4f}")

classifier_model.eval()
os.makedirs(OUTPUT_DIR_CLASSIFIER, exist_ok=True)
torch.save(classifier_model.state_dict(), os.path.join(OUTPUT_DIR_CLASSIFIER, "label_attention_classifier.pt"))
classifier_tokenizer.save_pretrained(OUTPUT_DIR_CLASSIFIER)
print(f"Classifier saved to {OUTPUT_DIR_CLASSIFIER}")


@torch.no_grad()
def classifier_probs_for(fact_text):
    enc = classifier_tokenizer(fact_text, truncation=True, padding="max_length",
                                max_length=CLASSIFIER_MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    _, probs = classifier_model(enc["input_ids"], enc["attention_mask"])
    probs = probs.squeeze(0).cpu().numpy()
    return {idx_to_label[i]: float(probs[i]) for i in range(num_classifier_labels)}

# =============================================================================
# STEP 6: IPC RETRIEVAL BRANCH
#   IPC Knowledge Base -> BM25 + Cosine Similarity -> Retrieval Score
#   (for 575 IPC sections)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)")
print("=" * 70)

retrieval_tokenizer = classifier_tokenizer
retrieval_encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

print("Building BM25 index over the IPC Knowledge Base ...")
catalog_texts = [ipc_catalog[norm_to_raw[sec]] for sec in all_section_codes]
bm25_index = BM25Okapi([_bm25_tokenize(t) for t in catalog_texts])


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def encode_texts(texts, max_len, batch_size=32):
    embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = retrieval_tokenizer(batch, truncation=True, padding=True, max_length=max_len,
                                   return_tensors="pt").to(DEVICE)
        out = retrieval_encoder(**enc).last_hidden_state
        pooled = F.normalize(mean_pool(out, enc["attention_mask"]), p=2, dim=-1)
        embeds.append(pooled.cpu())
    return torch.cat(embeds, dim=0) if embeds else torch.zeros((0, bert_hidden_size))


print("Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...")
catalog_embeddings = encode_texts(catalog_texts, RETRIEVAL_MAX_TOKEN_LEN)
print(f"Catalog embedding matrix: {tuple(catalog_embeddings.shape)}")


def retrieval_scores_for(fact_text):
    """BM25 + Cosine Similarity -> one fused Retrieval Score per of the 575
    IPC sections (document-level query against the whole fact text)."""
    fact_emb = encode_texts([fact_text], RETRIEVAL_MAX_TOKEN_LEN)
    cosine_scores = (fact_emb @ catalog_embeddings.t()).squeeze(0).numpy()
    bm25_scores = bm25_index.get_scores(_bm25_tokenize(fact_text))

    bm25_norm = _min_max_norm(bm25_scores)
    cosine_norm = _min_max_norm(cosine_scores)
    fused = RETRIEVAL_BM25_WEIGHT * bm25_norm + RETRIEVAL_COSINE_WEIGHT * cosine_norm
    return {all_section_codes[j]: float(fused[j]) for j in range(n_sections)}

# =============================================================================
# STEP 7: CALIBRATION ON THE VALIDATION SET
#   Per-classifier-class threshold, searched to maximize each class's own
#   val F1 -- and this time it's actually USED at prediction time (Step 8).
# =============================================================================
print("\n" + "=" * 70)
print("STEP 7: Calibrating per-class classifier thresholds on VAL")
print("=" * 70)

val_gold = {d["doc_id"]: gold_sections_of(d) for d in val_docs}
val_probs_matrix = np.zeros((len(val_docs), num_classifier_labels))
for i, d in enumerate(val_docs):
    probs = classifier_probs_for(d["fact"])
    for j, lab in enumerate(classifier_label_list):
        val_probs_matrix[i, j] = probs[lab]

val_targets_matrix = np.zeros_like(val_probs_matrix)
for i, d in enumerate(val_docs):
    gold = set(val_gold[d["doc_id"]])
    for j, lab in enumerate(classifier_label_list):
        val_targets_matrix[i, j] = 1.0 if lab in gold else 0.0

classifier_thresholds = {}
grid = np.arange(THRESHOLD_SEARCH_MIN, THRESHOLD_SEARCH_MAX + 1e-9, THRESHOLD_SEARCH_STEP)
for j, lab in enumerate(classifier_label_list):
    y_true = val_targets_matrix[:, j]
    if y_true.sum() == 0:
        classifier_thresholds[lab] = DEFAULT_CLASSIFIER_THRESHOLD
        continue
    best_t, best_f1 = DEFAULT_CLASSIFIER_THRESHOLD, -1.0
    for t in grid:
        y_pred = (val_probs_matrix[:, j] >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    classifier_thresholds[lab] = best_t
print(f"Calibrated {len(classifier_thresholds)} per-class thresholds: {classifier_thresholds}")

# =============================================================================
# STEP 8: SCORE NORMALIZATION AND FUSION -> THRESHOLDED STATUTE PREDICTION
#   (Evidence Sentence Retrieval and LLM Reasoning boxes removed)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 8: Fusion -> thresholded multi-label statute prediction")
print("=" * 70)


def predict_statutes(fact_text):
    cls_probs = classifier_probs_for(fact_text)
    retr_scores = retrieval_scores_for(fact_text)

    predicted = set()

    # (a) classifier branch: multi-label decision via each class's OWN
    #     calibrated threshold -- this is the actual fix for Macro-F1.
    for lab, p in cls_probs.items():
        if p >= classifier_thresholds[lab]:
            predicted.add(lab)

    # (b) retrieval branch: only add a retrieval-only section if it clears a
    #     conservative fixed bar (guards against the old spam-prediction bug)
    if ENABLE_RETRIEVAL_ONLY_PREDICTIONS:
        for sec, score in retr_scores.items():
            if sec not in label_to_idx and score >= RETRIEVAL_ONLY_THRESHOLD:
                predicted.add(sec)

    # (c) never emit an empty prediction: fall back to the single best fused
    #     candidate over the full 575-section space.
    if not predicted:
        cls_vector = np.zeros(n_sections)
        for lab, p in cls_probs.items():
            cls_vector[section_index[lab]] = p
        retr_vector = np.array([retr_scores[sec] for sec in all_section_codes])
        fused = (FUSION_CLASSIFIER_WEIGHT * _min_max_norm(cls_vector)
                 + FUSION_RETRIEVAL_WEIGHT * _min_max_norm(retr_vector))
        predicted.add(all_section_codes[int(np.argmax(fused))])

    return sorted(predicted)

# =============================================================================
# STEP 9: RUN ON THE HELD-OUT TEST SET
# =============================================================================
print("\n" + "=" * 70)
print(f"STEP 9: Predicting on {len(test_docs)} held-out TEST documents")
print("=" * 70)

all_predictions = {}
for d in test_docs:
    pred_secs = predict_statutes(d["fact"])
    all_predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": pred_secs}
    print(f"{d['doc_id']} -> pred={pred_secs} | gold={gold_sections_of(d)}")

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for rec in all_predictions.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"\nPredictions saved to: {PREDICTIONS_PATH}")

# =============================================================================
# STEP 10: EVALUATION -- Macro-F1 / Micro-F1 / Accuracy
# =============================================================================
print("\n" + "=" * 70)
print("STEP 10: Evaluation against gold labels")
print("=" * 70)


def compute_classification_metrics(test_docs, predictions):
    gold_labels = [gold_sections_of(d) for d in test_docs]
    pred_labels = [predictions[d["doc_id"]]["statute"] for d in test_docs]
    all_labels = sorted(set(l for labels in (gold_labels + pred_labels) for l in labels))
    mlb = MultiLabelBinarizer(classes=all_labels)
    y_true = mlb.fit_transform(gold_labels)
    y_pred = mlb.transform(pred_labels)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    exact_matches = sum(
        1 for d in test_docs
        if set(predictions[d["doc_id"]]["statute"]) == set(gold_sections_of(d))
    )
    accuracy = exact_matches / len(test_docs)
    return macro_f1, micro_f1, accuracy


macro_f1, micro_f1, accuracy = compute_classification_metrics(test_docs, all_predictions)
total = float(np.mean([macro_f1, micro_f1, accuracy]))

print("\nPerformance report (statute prediction only)")
print(f"{'Macro-F1':>10s}  {'Micro-F1':>10s}  {'Accuracy':>10s}  {'Total':>10s}")
print(f"{macro_f1:>10.4f}  {micro_f1:>10.4f}  {accuracy:>10.4f}  {total:>10.4f}")

# =============================================================================
# STEP 11: SIDE-BY-SIDE COMPARISON CSV
# =============================================================================
print("\n" + "=" * 70)
print("STEP 11: Writing predicted-vs-gold comparison CSV")
print("=" * 70)


def write_comparison_csv(test_docs, predictions, path):
    import csv
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "doc_id", "fact_snippet", "gold_sections", "predicted_sections",
            "correct_sections", "missed_sections", "extra_sections", "exact_match",
        ])
        for d in test_docs:
            gold = set(gold_sections_of(d))
            pred = set(predictions[d["doc_id"]]["statute"])
            fact_snippet = (d.get("fact", "") or "")[:150].replace("\n", " ")
            writer.writerow([
                d["doc_id"], fact_snippet,
                "; ".join(sorted(gold)), "; ".join(sorted(pred)),
                "; ".join(sorted(gold & pred)), "; ".join(sorted(gold - pred)), "; ".join(sorted(pred - gold)),
                "YES" if gold == pred else "NO",
            ])
    print(f"Wrote predicted-vs-gold comparison to: {path}")


write_comparison_csv(test_docs, all_predictions, COMPARISON_PATH)
print("\nDONE.")

Torch: 2.14.0+cu130 | CUDA available: True
Device: cuda
STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits
Loaded 575 official IPC sections from ipc_sections_clean.json
Total docs: 525 | Train: 378 | Val: 42 | Test: 105

STEP 4: Building the hybrid label space (classifier vs retrieval-only)
7 distinct gold sections seen in TRAIN docs.
-> 7 kept as CLASSIFIER classes (frequency >= 3).
-> remaining 568 of 575 sections are RETRIEVAL-ONLY.

STEP 5: Supervised Classification Branch (InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[classifier] epoch 1/8 -- avg loss: 1.9109
[classifier] epoch 2/8 -- avg loss: 1.6098
[classifier] epoch 3/8 -- avg loss: 1.2695
[classifier] epoch 4/8 -- avg loss: 1.0798
[classifier] epoch 5/8 -- avg loss: 0.9288
[classifier] epoch 6/8 -- avg loss: 0.8245
[classifier] epoch 7/8 -- avg loss: 0.7450
[classifier] epoch 8/8 -- avg loss: 0.7028
Classifier saved to ./classifier_out

STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building BM25 index over the IPC Knowledge Base ...
Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...
Catalog embedding matrix: (575, 768)

STEP 7: Calibrating per-class classifier thresholds on VAL
Calibrated 7 per-class thresholds: {'IPC 147': 0.18, 'IPC 201': 0.1, 'IPC 302': 0.22, 'IPC 376': 0.34, 'IPC 420': 0.22, 'IPC 498A': 0.32, 'IPC 506': 0.08}

STEP 8: Fusion -> thresholded multi-label statute prediction

STEP 9: Predicting on 105 held-out TEST documents
2002.INSC.274.txt -> pred=['IPC 375', 'IPC 376'] | gold=['IPC 376']
1999.INSC.378.txt -> pred=['IPC 113', 'IPC 154', 'IPC 155', 'IPC 156', 'IPC 354C', 'IPC 354D', 'IPC 375', 'IPC 376C', 'IPC 404', 'IPC 489E', 'IPC 498A', 'IPC 90'] | gold=['IPC 201', 'IPC 302']
2012.INSC.512.txt -> pred=['IPC 100', 'IPC 101', 'IPC 102', 'IPC 113', 'IPC 126', 'IPC 133', 'IPC 134', 'IPC 154', 'IPC 155', 'IPC 156', 'IPC 171C', 'IPC 201', 'IPC 249', 'IPC 253', 'IPC 297', 'IPC 298', 'IPC 302', 'IPC 318', 'IPC 324', 'IPC 326', 

In [4]:
"""
ipc_hybrid_statute_only.py
==========================
Simplified / fast version of the Classify-Retrieve-Evidence-Reason pipeline:
keeps ONLY the two branches that decide which statutes apply, drops the
Evidence Sentence Retrieval box and the LLM-Based Reasoning box entirely.

    Case Facts -> [Supervised Classification Branch] + [IPC Retrieval Branch]
               -> Score Normalization and Fusion
               -> THRESHOLDED statute prediction (multi-label decision)
               -> Predicted IPC sections

WHY THE PREVIOUS RUN SCORED MACRO-F1 = 0.14 (root cause + fix)
----------------------------------------------------------------
The previous `predict_document_hybrid` always emitted the top-TOP_K_CANDIDATES
(5) fused candidates as the FINAL prediction for every document (the LLM
reasoning stage was supposed to filter those 5 down, but with reasoning
removed nothing filtered them anymore). Since 568 of 575 IPC sections are
retrieval-only "noise" sections that almost never are actually correct, 3-4
of those 5 candidates per document were usually wrong. Each wrong section
becomes its OWN class with F1 = 0 in a macro-average, so ~30+ distinct
spuriously-predicted classes dragged Macro-F1 down to 0.14 even though
Micro-F1 (dominated by the few frequent, mostly-correct classes) still
looked fine at 0.53.

THE FIX in this version:
  1. The classifier's calibrated per-class thresholds (Step 7) are now
     ACTUALLY USED to decide the predicted set: a section from the 7
     classifier classes is predicted iff its softmax probability clears its
     own calibrated threshold -- true multi-label thresholding, not "always
     top-K".
  2. A retrieval-only section (one of the other 568) is only added to the
     prediction if its fused retrieval score clears a conservative fixed
     bar (RETRIEVAL_ONLY_THRESHOLD, default 0.85) -- this is what stops the
     old spam-prediction bug from reappearing. Since retrieval-only
     sections rarely have reliable supervision signal here, this is a
     high, deliberately conservative bar rather than something grid-searched
     to a degenerate low value.
  3. If nothing clears any threshold, we fall back to the single best fused
     candidate (never emit an empty prediction) -- this is what keeps
     Accuracy/Micro-F1 sane for docs sitting right at the decision boundary.

Everything else (BiLSTM + label-wise attention classifier, BM25 + cosine
retrieval, min-max fusion) is UNCHANGED from before -- same architecture,
just a corrected decision rule and the evidence/LLM stages removed for
speed.

Expected input files (unchanged):
  - task1.jsonl            : one JSON object per line, each with at least
                              {"doc_id": ..., "fact": ..., "statute": [...]}
  - ipc_sections_clean.json: [{"section": "302", "title": ..., "text": ...}, ...]
"""

# =============================================================================
# STEP 0: DEPENDENCIES
# =============================================================================
import subprocess
import sys


def ensure_packages():
    import importlib
    pkgs = {
        "torch": "torch",
        "transformers": "transformers",
        "scikit-learn": "sklearn",
        "numpy": "numpy",
        "rank_bm25": "rank_bm25",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing '{pip_name}' ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import json
import os
import re
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from rank_bm25 import BM25Okapi

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# =============================================================================
# STEP 1: CONFIG
# =============================================================================
TASK1_PATH = "task1.jsonl"
IPC_KB_PATH = "ipc_sections_clean.json"
OUTPUT_DIR_CLASSIFIER = "./classifier_out"
PREDICTIONS_PATH = "predictions_statute_only.jsonl"
COMPARISON_PATH = "comparison_pred_vs_gold.csv"

RANDOM_SEED = 42
TEST_FRACTION = 0.20
VAL_FRACTION = 0.10

MODEL_NAME = "law-ai/InLegalBERT"

# --- Supervised Classification Branch: InLegalBERT -> BiLSTM -> label-wise
#     Attention -> Linear & Softmax -> 7 IPC probabilities ---
MIN_CLASSIFIER_LABEL_FREQ = 3
CLASSIFIER_MAX_LENGTH = 384
CLASSIFIER_BATCH_SIZE = 8
CLASSIFIER_EPOCHS = 8
CLASSIFIER_LR = 2e-5
BILSTM_HIDDEN = 256
ATTN_DIM = 200

# --- IPC Retrieval Branch: BM25 + Cosine Similarity -> Retrieval Score ---
RETRIEVAL_MAX_TOKEN_LEN = 256
RETRIEVAL_BM25_WEIGHT = 0.5
RETRIEVAL_COSINE_WEIGHT = 0.5

# --- Score Normalization and Fusion (used only for the top-1 fallback now) ---
FUSION_CLASSIFIER_WEIGHT = 0.55
FUSION_RETRIEVAL_WEIGHT = 0.45

# --- Calibration: per-class softmax-probability threshold, searched on VAL
#     to maximize each class's own F1 -- THIS is what actually gates the
#     final prediction now (previously computed but unused). ---
THRESHOLD_SEARCH_MIN = 0.02
THRESHOLD_SEARCH_MAX = 0.60
THRESHOLD_SEARCH_STEP = 0.02
DEFAULT_CLASSIFIER_THRESHOLD = 0.15

# --- Retrieval-only sections (568 of 575) --------------------------------
# BUG FOUND AND FIXED HERE: retrieval_scores_for() min-max normalizes scores
# PER DOCUMENT (it stretches that document's own top score up to 1.0). That
# is fine for ranking sections *within* one document, but it is NOT a scale
# comparable across documents -- so comparing it against a fixed global
# threshold (the old RETRIEVAL_ONLY_THRESHOLD=0.85) pushed dozens of
# sections per document above 0.85 purely from the rescaling, which is
# exactly why the last run predicted 30-90 sections per document and
# Macro-F1 collapsed to 0.03. This dataset also only ever has 7 distinct
# gold sections, so the other 568 retrieval-only sections have NO real
# calibration signal to begin with. Fix: retrieval-only sections are simply
# not used to add predictions by default. Flip this to True only after you
# have re-derived a globally (not per-document) comparable retrieval score
# and re-calibrated its threshold on the validation set the same rigorous
# way the classifier thresholds are calibrated in Step 7.
ENABLE_RETRIEVAL_ONLY_PREDICTIONS = False
RETRIEVAL_ONLY_THRESHOLD = 0.85  # unused while the flag above is False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# =============================================================================
# STEP 2: SHARED TEXT UTILITIES
# =============================================================================
def normalize_ipc_label(label):
    label = str(label).strip()
    m = re.search(r"(\d+[A-Za-z]*)", label)
    if not m:
        return None
    return f"IPC {m.group(1).upper()}"


def load_jsonl_ordered(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_ipc_catalog(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    catalog = {}
    for entry in raw:
        code = str(entry.get("section", "")).strip()
        if not code:
            continue
        title = str(entry.get("title", "")).strip()
        text = str(entry.get("text", "")).strip()
        catalog[code] = f"{title}. {text}" if title else text
    return catalog, sorted(catalog.keys())


def gold_sections_of(doc):
    return sorted({s for s in (normalize_ipc_label(g) for g in doc.get("statute", [])) if s})


def _bm25_tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def _min_max_norm(arr):
    arr = np.asarray(arr, dtype=np.float64)
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-9:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

# =============================================================================
# STEP 3: LOAD DATA + SPLIT INTO TRAIN / VAL / TEST
# =============================================================================
print("=" * 70)
print("STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits")
print("=" * 70)

ipc_catalog, all_section_codes_raw = load_ipc_catalog(IPC_KB_PATH)
print(f"Loaded {len(all_section_codes_raw)} official IPC sections from {IPC_KB_PATH}")

norm_to_raw = {normalize_ipc_label(c): c for c in all_section_codes_raw}
all_section_codes = sorted(norm_to_raw.keys())
section_index = {sec: i for i, sec in enumerate(all_section_codes)}
n_sections = len(all_section_codes)

all_docs = load_jsonl_ordered(TASK1_PATH)
random.Random(RANDOM_SEED).shuffle(all_docs)

n = len(all_docs)
n_test = max(1, int(n * TEST_FRACTION))
n_val = max(1, int((n - n_test) * VAL_FRACTION))
test_docs = all_docs[:n_test]
val_docs = all_docs[n_test:n_test + n_val]
train_docs = all_docs[n_test + n_val:]
print(f"Total docs: {n} | Train: {len(train_docs)} | Val: {len(val_docs)} | Test: {len(test_docs)}")

# =============================================================================
# STEP 4: HYBRID LABEL SPACE (7 classifier classes vs retrieval-only)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: Building the hybrid label space (classifier vs retrieval-only)")
print("=" * 70)

doc_label_counts = Counter()
for d in train_docs:
    for s in gold_sections_of(d):
        doc_label_counts[s] += 1

classifier_label_list = sorted([s for s, c in doc_label_counts.items() if c >= MIN_CLASSIFIER_LABEL_FREQ])
label_to_idx = {s: i for i, s in enumerate(classifier_label_list)}
idx_to_label = {i: s for s, i in label_to_idx.items()}
num_classifier_labels = len(classifier_label_list)

print(f"{len(doc_label_counts)} distinct gold sections seen in TRAIN docs.")
print(f"-> {num_classifier_labels} kept as CLASSIFIER classes (frequency >= {MIN_CLASSIFIER_LABEL_FREQ}).")
print(f"-> remaining {n_sections - num_classifier_labels} of {n_sections} sections are RETRIEVAL-ONLY.")

# =============================================================================
# STEP 5: SUPERVISED CLASSIFICATION BRANCH
#   InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax
#   -> 7 IPC probabilities
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: Supervised Classification Branch "
      "(InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)")
print("=" * 70)

classifier_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_encoder = AutoModel.from_pretrained(MODEL_NAME)
bert_hidden_size = bert_encoder.config.hidden_size


class LabelWiseAttentionClassifier(nn.Module):
    """InLegalBERT -> BiLSTM -> label-wise attention -> Linear & Softmax."""

    def __init__(self, encoder, hidden_size, num_labels, lstm_hidden=BILSTM_HIDDEN, attn_dim=ATTN_DIM):
        super().__init__()
        self.encoder = encoder
        self.bilstm = nn.LSTM(hidden_size, lstm_hidden, batch_first=True, bidirectional=True)
        lstm_out_dim = lstm_hidden * 2
        self.attn_W = nn.Linear(lstm_out_dim, attn_dim, bias=False)
        self.attn_U = nn.Linear(attn_dim, num_labels, bias=False)
        self.label_weight = nn.Parameter(torch.randn(num_labels, lstm_out_dim) * 0.01)
        self.label_bias = nn.Parameter(torch.zeros(num_labels))
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.bilstm(enc_out)
        u = torch.tanh(self.attn_W(lstm_out))
        scores = self.attn_U(u)
        pad_mask = (~attention_mask.bool()).unsqueeze(-1)
        scores = scores.masked_fill(pad_mask, float("-inf"))
        alpha = torch.softmax(scores, dim=1)
        context = torch.einsum("btl,bth->blh", alpha, lstm_out)
        logits = torch.einsum("blh,lh->bl", context, self.label_weight) + self.label_bias
        probs = torch.softmax(logits, dim=-1)
        return logits, probs


classifier_model = LabelWiseAttentionClassifier(bert_encoder, bert_hidden_size, num_classifier_labels).to(DEVICE)


class StatuteDataset(Dataset):
    def __init__(self, records, label_to_idx, tokenizer, max_length):
        self.records = records
        self.label_to_idx = label_to_idx
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(rec["fact"], truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        target = torch.zeros(len(self.label_to_idx))
        for s in gold_sections_of(rec):
            if s in self.label_to_idx:
                target[self.label_to_idx[s]] = 1.0
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": target,
        }


def soft_target_cross_entropy(logits, multi_hot_targets):
    """Cross-entropy against a NORMALIZED multi-hot target so the single
    softmax head is still correctly supervised for multi-label documents."""
    row_sums = multi_hot_targets.sum(dim=-1, keepdim=True)
    safe_targets = torch.where(row_sums > 0, multi_hot_targets / row_sums.clamp(min=1e-9),
                                torch.full_like(multi_hot_targets, 1.0 / multi_hot_targets.size(-1)))
    log_probs = F.log_softmax(logits, dim=-1)
    return -(safe_targets * log_probs).sum(dim=-1).mean()


train_ds = StatuteDataset(train_docs, label_to_idx, classifier_tokenizer, CLASSIFIER_MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=CLASSIFIER_LR)
total_steps = max(1, len(train_loader) * CLASSIFIER_EPOCHS)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                             num_training_steps=total_steps)

classifier_model.train()
for epoch in range(CLASSIFIER_EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits, probs = classifier_model(input_ids, attention_mask)
        loss = soft_target_cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"[classifier] epoch {epoch + 1}/{CLASSIFIER_EPOCHS} -- avg loss: {total_loss / max(1, len(train_loader)):.4f}")

classifier_model.eval()
os.makedirs(OUTPUT_DIR_CLASSIFIER, exist_ok=True)
torch.save(classifier_model.state_dict(), os.path.join(OUTPUT_DIR_CLASSIFIER, "label_attention_classifier.pt"))
classifier_tokenizer.save_pretrained(OUTPUT_DIR_CLASSIFIER)
print(f"Classifier saved to {OUTPUT_DIR_CLASSIFIER}")


@torch.no_grad()
def classifier_probs_for(fact_text):
    enc = classifier_tokenizer(fact_text, truncation=True, padding="max_length",
                                max_length=CLASSIFIER_MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    _, probs = classifier_model(enc["input_ids"], enc["attention_mask"])
    probs = probs.squeeze(0).cpu().numpy()
    return {idx_to_label[i]: float(probs[i]) for i in range(num_classifier_labels)}

# =============================================================================
# STEP 6: IPC RETRIEVAL BRANCH
#   IPC Knowledge Base -> BM25 + Cosine Similarity -> Retrieval Score
#   (for 575 IPC sections)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)")
print("=" * 70)

retrieval_tokenizer = classifier_tokenizer
retrieval_encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

print("Building BM25 index over the IPC Knowledge Base ...")
catalog_texts = [ipc_catalog[norm_to_raw[sec]] for sec in all_section_codes]
bm25_index = BM25Okapi([_bm25_tokenize(t) for t in catalog_texts])


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def encode_texts(texts, max_len, batch_size=32):
    embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = retrieval_tokenizer(batch, truncation=True, padding=True, max_length=max_len,
                                   return_tensors="pt").to(DEVICE)
        out = retrieval_encoder(**enc).last_hidden_state
        pooled = F.normalize(mean_pool(out, enc["attention_mask"]), p=2, dim=-1)
        embeds.append(pooled.cpu())
    return torch.cat(embeds, dim=0) if embeds else torch.zeros((0, bert_hidden_size))


print("Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...")
catalog_embeddings = encode_texts(catalog_texts, RETRIEVAL_MAX_TOKEN_LEN)
print(f"Catalog embedding matrix: {tuple(catalog_embeddings.shape)}")


def retrieval_scores_for(fact_text):
    """BM25 + Cosine Similarity -> one fused Retrieval Score per of the 575
    IPC sections (document-level query against the whole fact text)."""
    fact_emb = encode_texts([fact_text], RETRIEVAL_MAX_TOKEN_LEN)
    cosine_scores = (fact_emb @ catalog_embeddings.t()).squeeze(0).numpy()
    bm25_scores = bm25_index.get_scores(_bm25_tokenize(fact_text))

    bm25_norm = _min_max_norm(bm25_scores)
    cosine_norm = _min_max_norm(cosine_scores)
    fused = RETRIEVAL_BM25_WEIGHT * bm25_norm + RETRIEVAL_COSINE_WEIGHT * cosine_norm
    return {all_section_codes[j]: float(fused[j]) for j in range(n_sections)}

# =============================================================================
# STEP 7: CALIBRATION ON THE VALIDATION SET
#   Per-classifier-class threshold, searched to maximize each class's own
#   val F1 -- and this time it's actually USED at prediction time (Step 8).
# =============================================================================
print("\n" + "=" * 70)
print("STEP 7: Calibrating per-class classifier thresholds on VAL")
print("=" * 70)

val_gold = {d["doc_id"]: gold_sections_of(d) for d in val_docs}
val_probs_matrix = np.zeros((len(val_docs), num_classifier_labels))
for i, d in enumerate(val_docs):
    probs = classifier_probs_for(d["fact"])
    for j, lab in enumerate(classifier_label_list):
        val_probs_matrix[i, j] = probs[lab]

val_targets_matrix = np.zeros_like(val_probs_matrix)
for i, d in enumerate(val_docs):
    gold = set(val_gold[d["doc_id"]])
    for j, lab in enumerate(classifier_label_list):
        val_targets_matrix[i, j] = 1.0 if lab in gold else 0.0

classifier_thresholds = {}
grid = np.arange(THRESHOLD_SEARCH_MIN, THRESHOLD_SEARCH_MAX + 1e-9, THRESHOLD_SEARCH_STEP)
for j, lab in enumerate(classifier_label_list):
    y_true = val_targets_matrix[:, j]
    if y_true.sum() == 0:
        classifier_thresholds[lab] = DEFAULT_CLASSIFIER_THRESHOLD
        continue
    best_t, best_f1 = DEFAULT_CLASSIFIER_THRESHOLD, -1.0
    for t in grid:
        y_pred = (val_probs_matrix[:, j] >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    classifier_thresholds[lab] = best_t
print(f"Calibrated {len(classifier_thresholds)} per-class thresholds: {classifier_thresholds}")

# =============================================================================
# STEP 8: SCORE NORMALIZATION AND FUSION -> THRESHOLDED STATUTE PREDICTION
#   (Evidence Sentence Retrieval and LLM Reasoning boxes removed)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 8: Fusion -> thresholded multi-label statute prediction")
print("=" * 70)


def predict_statutes(fact_text):
    cls_probs = classifier_probs_for(fact_text)
    retr_scores = retrieval_scores_for(fact_text)

    predicted = set()

    # (a) classifier branch: multi-label decision via each class's OWN
    #     calibrated threshold -- this is the actual fix for Macro-F1.
    for lab, p in cls_probs.items():
        if p >= classifier_thresholds[lab]:
            predicted.add(lab)

    # (b) retrieval branch: only add a retrieval-only section if it clears a
    #     conservative fixed bar. DISABLED BY DEFAULT (see the config
    #     comment above ENABLE_RETRIEVAL_ONLY_PREDICTIONS) -- the retrieval
    #     score is min-max normalized PER DOCUMENT, so it is not on a scale
    #     comparable to a fixed threshold across documents, and this dataset
    #     has no real gold signal for the 568 retrieval-only sections to
    #     calibrate against anyway. Left in place (off) so the branch is
    #     still visibly part of the architecture / easy to re-enable once
    #     properly recalibrated.
    if ENABLE_RETRIEVAL_ONLY_PREDICTIONS:
        for sec, score in retr_scores.items():
            if sec not in label_to_idx and score >= RETRIEVAL_ONLY_THRESHOLD:
                predicted.add(sec)

    # (c) never emit an empty prediction: fall back to the classifier's own
    #     single best class (NOT a retrieval-influenced fused score -- using
    #     the fused vector here reintroduces the same per-document-relative-
    #     vs-fixed-threshold instability that broke the retrieval-only
    #     branch above).
    if not predicted:
        best_label = max(cls_probs, key=cls_probs.get)
        predicted.add(best_label)

    return sorted(predicted)

# =============================================================================
# STEP 9: RUN ON THE HELD-OUT TEST SET
# =============================================================================
print("\n" + "=" * 70)
print(f"STEP 9: Predicting on {len(test_docs)} held-out TEST documents")
print("=" * 70)

all_predictions = {}
for d in test_docs:
    pred_secs = predict_statutes(d["fact"])
    all_predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": pred_secs}
    print(f"{d['doc_id']} -> pred={pred_secs} | gold={gold_sections_of(d)}")

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for rec in all_predictions.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"\nPredictions saved to: {PREDICTIONS_PATH}")

# =============================================================================
# STEP 10: EVALUATION -- Macro-F1 / Micro-F1 / Accuracy
# =============================================================================
print("\n" + "=" * 70)
print("STEP 10: Evaluation against gold labels")
print("=" * 70)


def compute_classification_metrics(test_docs, predictions):
    gold_labels = [gold_sections_of(d) for d in test_docs]
    pred_labels = [predictions[d["doc_id"]]["statute"] for d in test_docs]
    all_labels = sorted(set(l for labels in (gold_labels + pred_labels) for l in labels))
    mlb = MultiLabelBinarizer(classes=all_labels)
    y_true = mlb.fit_transform(gold_labels)
    y_pred = mlb.transform(pred_labels)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    exact_matches = sum(
        1 for d in test_docs
        if set(predictions[d["doc_id"]]["statute"]) == set(gold_sections_of(d))
    )
    accuracy = exact_matches / len(test_docs)
    return macro_f1, micro_f1, accuracy


macro_f1, micro_f1, accuracy = compute_classification_metrics(test_docs, all_predictions)
total = float(np.mean([macro_f1, micro_f1, accuracy]))

print("\nPerformance report (statute prediction only)")
print(f"{'Macro-F1':>10s}  {'Micro-F1':>10s}  {'Accuracy':>10s}  {'Total':>10s}")
print(f"{macro_f1:>10.4f}  {micro_f1:>10.4f}  {accuracy:>10.4f}  {total:>10.4f}")

# =============================================================================
# STEP 11: SIDE-BY-SIDE COMPARISON CSV
# =============================================================================
print("\n" + "=" * 70)
print("STEP 11: Writing predicted-vs-gold comparison CSV")
print("=" * 70)


def write_comparison_csv(test_docs, predictions, path):
    import csv
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "doc_id", "fact_snippet", "gold_sections", "predicted_sections",
            "correct_sections", "missed_sections", "extra_sections", "exact_match",
        ])
        for d in test_docs:
            gold = set(gold_sections_of(d))
            pred = set(predictions[d["doc_id"]]["statute"])
            fact_snippet = (d.get("fact", "") or "")[:150].replace("\n", " ")
            writer.writerow([
                d["doc_id"], fact_snippet,
                "; ".join(sorted(gold)), "; ".join(sorted(pred)),
                "; ".join(sorted(gold & pred)), "; ".join(sorted(gold - pred)), "; ".join(sorted(pred - gold)),
                "YES" if gold == pred else "NO",
            ])
    print(f"Wrote predicted-vs-gold comparison to: {path}")


write_comparison_csv(test_docs, all_predictions, COMPARISON_PATH)
print("\nDONE.")

Torch: 2.14.0+cu130 | CUDA available: True
Device: cuda
STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits
Loaded 575 official IPC sections from ipc_sections_clean.json
Total docs: 525 | Train: 378 | Val: 42 | Test: 105

STEP 4: Building the hybrid label space (classifier vs retrieval-only)
7 distinct gold sections seen in TRAIN docs.
-> 7 kept as CLASSIFIER classes (frequency >= 3).
-> remaining 568 of 575 sections are RETRIEVAL-ONLY.

STEP 5: Supervised Classification Branch (InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[classifier] epoch 1/8 -- avg loss: 1.9119
[classifier] epoch 2/8 -- avg loss: 1.6369
[classifier] epoch 3/8 -- avg loss: 1.2935
[classifier] epoch 4/8 -- avg loss: 1.0614
[classifier] epoch 5/8 -- avg loss: 0.9138
[classifier] epoch 6/8 -- avg loss: 0.7874
[classifier] epoch 7/8 -- avg loss: 0.7220
[classifier] epoch 8/8 -- avg loss: 0.6860
Classifier saved to ./classifier_out

STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building BM25 index over the IPC Knowledge Base ...
Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...
Catalog embedding matrix: (575, 768)

STEP 7: Calibrating per-class classifier thresholds on VAL
Calibrated 7 per-class thresholds: {'IPC 147': 0.12000000000000001, 'IPC 201': 0.1, 'IPC 302': 0.30000000000000004, 'IPC 376': 0.24, 'IPC 420': 0.12000000000000001, 'IPC 498A': 0.22, 'IPC 506': 0.06}

STEP 8: Fusion -> thresholded multi-label statute prediction

STEP 9: Predicting on 105 held-out TEST documents
2002.INSC.274.txt -> pred=['IPC 376'] | gold=['IPC 376']
1999.INSC.378.txt -> pred=['IPC 376', 'IPC 506'] | gold=['IPC 201', 'IPC 302']
2012.INSC.512.txt -> pred=['IPC 201', 'IPC 302', 'IPC 506'] | gold=['IPC 498A']
1998.INSC.126.txt -> pred=['IPC 302'] | gold=['IPC 302']
2007.INSC.590.txt -> pred=['IPC 147', 'IPC 302'] | gold=['IPC 147']
2013.INSC.960.txt -> pred=['IPC 420', 'IPC 506'] | gold=['IPC 201']
2009.INSC.1130.txt -> pred=['IPC 376'] | gold=['IPC 376

In [5]:
"""
ipc_hybrid_full_pipeline_fixed.py
==================================
Full "Classify-Retrieve-Evidence-Reason" pipeline, built ON TOP OF the
CORRECTED statute-decision logic from ipc_hybrid_statute_only.py (the
version that fixed Macro-F1). This restores the two boxes that were
removed for speed -- Evidence Sentence Retrieval and LLM-Based Reasoning
Generation -- without touching the fixed decision logic that got Macro-F1
back to a sane value.

    Case Facts -> [Supervised Classification Branch] + [IPC Retrieval Branch]
               -> Score Normalization and Fusion -> IPC Candidate Ranking
               -> Evidence Sentence Retrieval for each candidate IPC
               -> LLM-Based Reasoning Generation (Qwen, CoT)
               -> Predicted IPC sections + Evidence Sentences + Explanation

IMPORTANT DESIGN DECISION (read this before changing anything):
-----------------------------------------------------------------
Two bugs already cost you a lot of Macro-F1 in earlier iterations:
  1. Always emitting the top-K fused candidates as the final prediction
     (no filtering) -> Macro-F1 0.14 (spam from retrieval-only sections).
  2. Comparing a PER-DOCUMENT min-max-normalized retrieval score against a
     FIXED global threshold -> Macro-F1 0.03 (even worse spam).
The fix that got you back to a sane Macro-F1 was: predicted sections come
ONLY from the classifier branch's calibrated per-class thresholds (with a
classifier-only single-best fallback so nothing is ever empty). That
decision rule is UNCHANGED here -- `predict_statutes()` is copied verbatim.

The "IPC Candidate Ranking" fed into Evidence Retrieval + LLM Reasoning in
this version is therefore simply the ALREADY-DECIDED predicted section set
(normally 1, occasionally 2-3 sections), not a blind top-5. The LLM is used
to explain *why* each already-decided section applies (evidence sentences +
a CoT explanation) -- its own "applies" verdict is recorded for your
inspection but does NOT remove a section from the official prediction,
so it cannot re-introduce the spam bug or silently change your Macro-F1.
If you want the LLM to be allowed to veto a prediction, see the comment
right above `predicted_with_reasoning` in Step 11.

Expected input files (unchanged):
  - task1.jsonl            : one JSON object per line, each with at least
                              {"doc_id": ..., "fact": ..., "statute": [...],
                               "explanation": {sentence: ipc_label, ...}}
  - ipc_sections_clean.json: [{"section": "302", "title": ..., "text": ...}, ...]
"""

# =============================================================================
# STEP 0: DEPENDENCIES
# =============================================================================
import subprocess
import sys


def ensure_packages():
    import importlib
    pkgs = {
        "torch": "torch",
        "transformers": "transformers",
        "scikit-learn": "sklearn",
        "accelerate": "accelerate",
        "numpy": "numpy",
        "rank_bm25": "rank_bm25",
        "rouge_score": "rouge_score",
        "nltk": "nltk",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing '{pip_name}' ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)

    import nltk
    for res, pkg in [("tokenizers/punkt", "punkt"), ("tokenizers/punkt_tab", "punkt_tab"),
                      ("corpora/wordnet", "wordnet"), ("corpora/omw-1.4", "omw-1.4")]:
        try:
            nltk.data.find(res)
        except LookupError:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass


ensure_packages()

import json
import os
import re
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForCausalLM, get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from rank_bm25 import BM25Okapi
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# =============================================================================
# STEP 1: CONFIG
# =============================================================================
TASK1_PATH = "task1.jsonl"
IPC_KB_PATH = "ipc_sections_clean.json"
OUTPUT_DIR_CLASSIFIER = "./classifier_out"
PREDICTIONS_PATH = "predictions_full_pipeline.jsonl"
COMPARISON_PATH = "comparison_pred_vs_gold.csv"

RANDOM_SEED = 42
TEST_FRACTION = 0.20
VAL_FRACTION = 0.10

MODEL_NAME = "law-ai/InLegalBERT"

# --- Supervised Classification Branch ---
MIN_CLASSIFIER_LABEL_FREQ = 3
CLASSIFIER_MAX_LENGTH = 384
CLASSIFIER_BATCH_SIZE = 8
CLASSIFIER_EPOCHS = 8
CLASSIFIER_LR = 2e-5
BILSTM_HIDDEN = 256
ATTN_DIM = 200

# --- IPC Retrieval Branch ---
RETRIEVAL_MAX_TOKEN_LEN = 256
RETRIEVAL_BM25_WEIGHT = 0.5
RETRIEVAL_COSINE_WEIGHT = 0.5

# --- Calibration (this is what actually decides the predicted sections) ---
THRESHOLD_SEARCH_MIN = 0.02
THRESHOLD_SEARCH_MAX = 0.60
THRESHOLD_SEARCH_STEP = 0.02
DEFAULT_CLASSIFIER_THRESHOLD = 0.15

# --- Retrieval-only sections: OFF by default -- see the long comment in
#     Step 8 (predict_statutes) for why. Left here only for completeness. ---
ENABLE_RETRIEVAL_ONLY_PREDICTIONS = False
RETRIEVAL_ONLY_THRESHOLD = 0.85  # unused while the flag above is False

# --- Evidence Sentence Retrieval for Each Candidate IPC ---
EVIDENCE_BM25_WEIGHT = 0.34
EVIDENCE_COSINE_WEIGHT = 0.33
EVIDENCE_CLS_WEIGHT = 0.33
TOP_M_EVIDENCE = 3

# --- LLM-Based Reasoning Generation (Qwen, CoT prompting) ---
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MAX_NEW_TOKENS = 300
LLM_TEMPERATURE = 0.2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# =============================================================================
# STEP 2: SHARED TEXT UTILITIES
# =============================================================================
_ABBREV_PATTERNS = [
    r"\bPW-?\d*\.", r"\bp\.m\.", r"\ba\.m\.", r"\bExt\.-?", r"\bRs\.",
    r"\bNo\.", r"\bSec\.", r"\bSection\.", r"\bvs\.", r"\bv\.", r"\bMr\.",
    r"\bMrs\.", r"\bDr\.", r"\bJ\.\)", r"\bi\.e\.", r"\be\.g\.", r"\bIPC\.",
    r"\bCrPC\.", r"\bHon'ble\.", r"\bU/s\.",
]
_PLACEHOLDER = "<<DOT_{}>>"


def split_sentences_with_spans(text):
    protected = text
    placeholders = {}
    for i, pat in enumerate(_ABBREV_PATTERNS):
        def _sub(m, i=i):
            key = _PLACEHOLDER.format(f"{i}_{len(placeholders)}")
            placeholders[key] = m.group(0)
            return key
        protected = re.sub(pat, _sub, protected)

    raw_sents = re.split(r"(?<=[.!?])\s+(?=[A-Z(\"\u2018\u201c])", protected)

    results = []
    cursor = 0
    for s in raw_sents:
        for key, val in placeholders.items():
            s = s.replace(key, val)
        s_stripped = s.strip()
        if not s_stripped:
            continue
        idx = text.find(s_stripped, cursor)
        if idx == -1:
            idx = text.find(s_stripped)
        if idx == -1:
            start, end = cursor, cursor + len(s_stripped)
        else:
            start, end = idx, idx + len(s_stripped)
        results.append((s_stripped, start, end))
        cursor = end
    return results


def split_sentences(text):
    return [s for s, _, _ in split_sentences_with_spans(text)]


def normalize_ipc_label(label):
    label = str(label).strip()
    m = re.search(r"(\d+[A-Za-z]*)", label)
    if not m:
        return None
    return f"IPC {m.group(1).upper()}"


def load_jsonl_ordered(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_ipc_catalog(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    catalog, titles = {}, {}
    for entry in raw:
        code = str(entry.get("section", "")).strip()
        if not code:
            continue
        title = str(entry.get("title", "")).strip()
        text = str(entry.get("text", "")).strip()
        catalog[code] = f"{title}. {text}" if title else text
        titles[code] = title
    return catalog, titles, sorted(catalog.keys())


def gold_sections_of(doc):
    return sorted({s for s in (normalize_ipc_label(g) for g in doc.get("statute", [])) if s})


def gold_explanation_reference(doc):
    """Reference text for ROUGE/BLEU/METEOR: gold (sentence -> IPC) pairs
    joined into one paragraph -- there is no other free-text gold
    'reasoning' available in task1.jsonl."""
    exp = doc.get("explanation", {}) or {}
    if not exp:
        return " ".join(split_sentences(doc.get("fact", ""))[:2])
    return " ".join(f"{sent.strip()} (=> {label})" for sent, label in exp.items())


def _bm25_tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def _min_max_norm(arr):
    arr = np.asarray(arr, dtype=np.float64)
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-9:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

# =============================================================================
# STEP 3: LOAD DATA + SPLIT INTO TRAIN / VAL / TEST
# =============================================================================
print("=" * 70)
print("STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits")
print("=" * 70)

ipc_catalog, ipc_titles, all_section_codes_raw = load_ipc_catalog(IPC_KB_PATH)
print(f"Loaded {len(all_section_codes_raw)} official IPC sections from {IPC_KB_PATH}")

norm_to_raw = {normalize_ipc_label(c): c for c in all_section_codes_raw}
all_section_codes = sorted(norm_to_raw.keys())
section_index = {sec: i for i, sec in enumerate(all_section_codes)}
n_sections = len(all_section_codes)

all_docs = load_jsonl_ordered(TASK1_PATH)
random.Random(RANDOM_SEED).shuffle(all_docs)

n = len(all_docs)
n_test = max(1, int(n * TEST_FRACTION))
n_val = max(1, int((n - n_test) * VAL_FRACTION))
test_docs = all_docs[:n_test]
val_docs = all_docs[n_test:n_test + n_val]
train_docs = all_docs[n_test + n_val:]
print(f"Total docs: {n} | Train: {len(train_docs)} | Val: {len(val_docs)} | Test: {len(test_docs)}")

# =============================================================================
# STEP 4: HYBRID LABEL SPACE (7 classifier classes vs retrieval-only)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: Building the hybrid label space (classifier vs retrieval-only)")
print("=" * 70)

doc_label_counts = Counter()
for d in train_docs:
    for s in gold_sections_of(d):
        doc_label_counts[s] += 1

classifier_label_list = sorted([s for s, c in doc_label_counts.items() if c >= MIN_CLASSIFIER_LABEL_FREQ])
label_to_idx = {s: i for i, s in enumerate(classifier_label_list)}
idx_to_label = {i: s for s, i in label_to_idx.items()}
num_classifier_labels = len(classifier_label_list)

print(f"{len(doc_label_counts)} distinct gold sections seen in TRAIN docs.")
print(f"-> {num_classifier_labels} kept as CLASSIFIER classes (frequency >= {MIN_CLASSIFIER_LABEL_FREQ}).")
print(f"-> remaining {n_sections - num_classifier_labels} of {n_sections} sections are RETRIEVAL-ONLY.")

# =============================================================================
# STEP 5: SUPERVISED CLASSIFICATION BRANCH
#   InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax
#   -> 7 IPC probabilities
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: Supervised Classification Branch "
      "(InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)")
print("=" * 70)

classifier_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_encoder = AutoModel.from_pretrained(MODEL_NAME)
bert_hidden_size = bert_encoder.config.hidden_size


class LabelWiseAttentionClassifier(nn.Module):
    """InLegalBERT -> BiLSTM -> label-wise attention -> Linear & Softmax.
    Returns `alpha` (per-label attention over tokens) too, so the Evidence
    Sentence Retrieval stage can reuse it as 'Classifier-Based Relevance'."""

    def __init__(self, encoder, hidden_size, num_labels, lstm_hidden=BILSTM_HIDDEN, attn_dim=ATTN_DIM):
        super().__init__()
        self.encoder = encoder
        self.bilstm = nn.LSTM(hidden_size, lstm_hidden, batch_first=True, bidirectional=True)
        lstm_out_dim = lstm_hidden * 2
        self.attn_W = nn.Linear(lstm_out_dim, attn_dim, bias=False)
        self.attn_U = nn.Linear(attn_dim, num_labels, bias=False)
        self.label_weight = nn.Parameter(torch.randn(num_labels, lstm_out_dim) * 0.01)
        self.label_bias = nn.Parameter(torch.zeros(num_labels))
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.bilstm(enc_out)
        u = torch.tanh(self.attn_W(lstm_out))
        scores = self.attn_U(u)
        pad_mask = (~attention_mask.bool()).unsqueeze(-1)
        scores = scores.masked_fill(pad_mask, float("-inf"))
        alpha = torch.softmax(scores, dim=1)
        context = torch.einsum("btl,bth->blh", alpha, lstm_out)
        logits = torch.einsum("blh,lh->bl", context, self.label_weight) + self.label_bias
        probs = torch.softmax(logits, dim=-1)
        return logits, probs, alpha


classifier_model = LabelWiseAttentionClassifier(bert_encoder, bert_hidden_size, num_classifier_labels).to(DEVICE)


class StatuteDataset(Dataset):
    def __init__(self, records, label_to_idx, tokenizer, max_length):
        self.records = records
        self.label_to_idx = label_to_idx
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(rec["fact"], truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        target = torch.zeros(len(self.label_to_idx))
        for s in gold_sections_of(rec):
            if s in self.label_to_idx:
                target[self.label_to_idx[s]] = 1.0
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": target,
        }


def soft_target_cross_entropy(logits, multi_hot_targets):
    row_sums = multi_hot_targets.sum(dim=-1, keepdim=True)
    safe_targets = torch.where(row_sums > 0, multi_hot_targets / row_sums.clamp(min=1e-9),
                                torch.full_like(multi_hot_targets, 1.0 / multi_hot_targets.size(-1)))
    log_probs = F.log_softmax(logits, dim=-1)
    return -(safe_targets * log_probs).sum(dim=-1).mean()


train_ds = StatuteDataset(train_docs, label_to_idx, classifier_tokenizer, CLASSIFIER_MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=CLASSIFIER_LR)
total_steps = max(1, len(train_loader) * CLASSIFIER_EPOCHS)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                             num_training_steps=total_steps)

classifier_model.train()
for epoch in range(CLASSIFIER_EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits, probs, _ = classifier_model(input_ids, attention_mask)
        loss = soft_target_cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"[classifier] epoch {epoch + 1}/{CLASSIFIER_EPOCHS} -- avg loss: {total_loss / max(1, len(train_loader)):.4f}")

classifier_model.eval()
os.makedirs(OUTPUT_DIR_CLASSIFIER, exist_ok=True)
torch.save(classifier_model.state_dict(), os.path.join(OUTPUT_DIR_CLASSIFIER, "label_attention_classifier.pt"))
classifier_tokenizer.save_pretrained(OUTPUT_DIR_CLASSIFIER)
print(f"Classifier saved to {OUTPUT_DIR_CLASSIFIER}")


@torch.no_grad()
def classifier_probs_for(fact_text):
    """Lightweight variant (no offsets) -- used for VAL calibration (Step 7)
    where only the 7 probabilities are needed, for speed."""
    enc = classifier_tokenizer(fact_text, truncation=True, padding="max_length",
                                max_length=CLASSIFIER_MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    _, probs, _ = classifier_model(enc["input_ids"], enc["attention_mask"])
    probs = probs.squeeze(0).cpu().numpy()
    return {idx_to_label[i]: float(probs[i]) for i in range(num_classifier_labels)}


@torch.no_grad()
def classifier_forward_for(fact_text):
    """Full variant (with offsets) -- used at test time so Evidence Sentence
    Retrieval can reuse the label-wise attention as 'Classifier-Based
    Relevance'."""
    enc = classifier_tokenizer(fact_text, truncation=True, padding="max_length",
                                max_length=CLASSIFIER_MAX_LENGTH, return_tensors="pt",
                                return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    _, probs, alpha = classifier_model(enc["input_ids"], enc["attention_mask"])
    probs = probs.squeeze(0).cpu().numpy()
    alpha = alpha.squeeze(0).cpu().numpy()
    attn_mask = enc["attention_mask"].squeeze(0).cpu().numpy()
    probs_by_label = {idx_to_label[i]: float(probs[i]) for i in range(num_classifier_labels)}
    return probs_by_label, alpha, offsets.numpy(), attn_mask


def classifier_attention_per_sentence(fact_text, label, alpha, offsets, attn_mask):
    """Aggregates the label-wise attention mass onto each sentence's
    character span -- this IS 'Classifier-Based Relevance' in the diagram."""
    if label not in label_to_idx:
        return None
    lab_idx = label_to_idx[label]
    sent_spans = split_sentences_with_spans(fact_text)
    if not sent_spans:
        return []
    scores = [0.0] * len(sent_spans)
    for t in range(len(offsets)):
        if attn_mask[t] == 0:
            continue
        tok_start, tok_end = int(offsets[t][0]), int(offsets[t][1])
        if tok_end <= tok_start:
            continue
        for si, (_, s_start, s_end) in enumerate(sent_spans):
            if tok_start < s_end and tok_end > s_start:
                scores[si] += float(alpha[t, lab_idx])
                break
    return scores

# =============================================================================
# STEP 6: IPC RETRIEVAL BRANCH
#   IPC Knowledge Base -> BM25 + Cosine Similarity -> Retrieval Score
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)")
print("=" * 70)

retrieval_tokenizer = classifier_tokenizer
retrieval_encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

print("Building BM25 index over the IPC Knowledge Base ...")
catalog_texts = [ipc_catalog[norm_to_raw[sec]] for sec in all_section_codes]
bm25_index = BM25Okapi([_bm25_tokenize(t) for t in catalog_texts])


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def encode_texts(texts, max_len, batch_size=32):
    embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = retrieval_tokenizer(batch, truncation=True, padding=True, max_length=max_len,
                                   return_tensors="pt").to(DEVICE)
        out = retrieval_encoder(**enc).last_hidden_state
        pooled = F.normalize(mean_pool(out, enc["attention_mask"]), p=2, dim=-1)
        embeds.append(pooled.cpu())
    return torch.cat(embeds, dim=0) if embeds else torch.zeros((0, bert_hidden_size))


print("Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...")
catalog_embeddings = encode_texts(catalog_texts, RETRIEVAL_MAX_TOKEN_LEN)
print(f"Catalog embedding matrix: {tuple(catalog_embeddings.shape)}")


def retrieval_scores_for(fact_text):
    """BM25 + Cosine Similarity -> fused Retrieval Score for all 575 IPC
    sections (PER-DOCUMENT min-max normalized -- fine for ranking WITHIN one
    document, not for comparing across documents against a fixed bar; see
    the Step 8 comment for why that distinction matters)."""
    fact_emb = encode_texts([fact_text], RETRIEVAL_MAX_TOKEN_LEN)
    cosine_scores = (fact_emb @ catalog_embeddings.t()).squeeze(0).numpy()
    bm25_scores = bm25_index.get_scores(_bm25_tokenize(fact_text))

    bm25_norm = _min_max_norm(bm25_scores)
    cosine_norm = _min_max_norm(cosine_scores)
    fused = RETRIEVAL_BM25_WEIGHT * bm25_norm + RETRIEVAL_COSINE_WEIGHT * cosine_norm
    return {all_section_codes[j]: float(fused[j]) for j in range(n_sections)}

# =============================================================================
# STEP 7: CALIBRATION ON THE VALIDATION SET
# =============================================================================
print("\n" + "=" * 70)
print("STEP 7: Calibrating per-class classifier thresholds on VAL")
print("=" * 70)

val_gold = {d["doc_id"]: gold_sections_of(d) for d in val_docs}
val_probs_matrix = np.zeros((len(val_docs), num_classifier_labels))
for i, d in enumerate(val_docs):
    probs = classifier_probs_for(d["fact"])
    for j, lab in enumerate(classifier_label_list):
        val_probs_matrix[i, j] = probs[lab]

val_targets_matrix = np.zeros_like(val_probs_matrix)
for i, d in enumerate(val_docs):
    gold = set(val_gold[d["doc_id"]])
    for j, lab in enumerate(classifier_label_list):
        val_targets_matrix[i, j] = 1.0 if lab in gold else 0.0

classifier_thresholds = {}
grid = np.arange(THRESHOLD_SEARCH_MIN, THRESHOLD_SEARCH_MAX + 1e-9, THRESHOLD_SEARCH_STEP)
for j, lab in enumerate(classifier_label_list):
    y_true = val_targets_matrix[:, j]
    if y_true.sum() == 0:
        classifier_thresholds[lab] = DEFAULT_CLASSIFIER_THRESHOLD
        continue
    best_t, best_f1 = DEFAULT_CLASSIFIER_THRESHOLD, -1.0
    for t in grid:
        y_pred = (val_probs_matrix[:, j] >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    classifier_thresholds[lab] = best_t
print(f"Calibrated {len(classifier_thresholds)} per-class thresholds: {classifier_thresholds}")

# =============================================================================
# STEP 8: SCORE NORMALIZATION AND FUSION -> IPC CANDIDATE RANKING
#   (this decision rule is UNCHANGED from the fixed statute-only version --
#   do not blindly take top-K here, that is the bug that broke Macro-F1
#   twice already)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 8: Fusion -> thresholded multi-label statute prediction")
print("=" * 70)


def predict_statutes(cls_probs, retr_scores):
    predicted = set()

    # (a) classifier branch: multi-label decision via each class's OWN
    #     calibrated threshold -- the actual fix for Macro-F1.
    for lab, p in cls_probs.items():
        if p >= classifier_thresholds[lab]:
            predicted.add(lab)

    # (b) retrieval-only sections: OFF by default. retr_scores is min-max
    #     normalized PER DOCUMENT, so comparing it to a fixed global bar
    #     across documents reintroduces the spam bug that gave Macro-F1 =
    #     0.03 last time. Only flip this on after re-deriving a globally
    #     comparable retrieval score and recalibrating on VAL like Step 7.
    if ENABLE_RETRIEVAL_ONLY_PREDICTIONS:
        for sec, score in retr_scores.items():
            if sec not in label_to_idx and score >= RETRIEVAL_ONLY_THRESHOLD:
                predicted.add(sec)

    # (c) never emit an empty prediction: fall back to the classifier's own
    #     single best class (not a retrieval-influenced fused score).
    if not predicted:
        predicted.add(max(cls_probs, key=cls_probs.get))

    return sorted(predicted)

# =============================================================================
# STEP 9: EVIDENCE SENTENCE RETRIEVAL FOR EACH CANDIDATE IPC
#   Sentence Scoring S = {S1..Sn} via BM25 + Cosine Similarity +
#   Classifier-Based Relevance -> Top-m Evidence Sentences
# =============================================================================
print("\n" + "=" * 70)
print("STEP 9: Evidence Sentence Retrieval for each predicted IPC")
print("=" * 70)


def evidence_sentences_for_candidate(fact_text, section, alpha, offsets, mask):
    sentences = split_sentences(fact_text)
    if not sentences:
        return []
    section_text = ipc_catalog[norm_to_raw[section]]

    sent_tokens = [_bm25_tokenize(s) for s in sentences]
    local_bm25 = BM25Okapi(sent_tokens) if sent_tokens else None
    bm25_scores = local_bm25.get_scores(_bm25_tokenize(section_text)) if local_bm25 else np.zeros(len(sentences))

    sent_embs = encode_texts(sentences, RETRIEVAL_MAX_TOKEN_LEN)
    section_emb = encode_texts([section_text], RETRIEVAL_MAX_TOKEN_LEN)
    cosine_scores = (sent_embs @ section_emb.t()).squeeze(-1).numpy()

    cls_scores = classifier_attention_per_sentence(fact_text, section, alpha, offsets, mask)

    bm25_norm = _min_max_norm(bm25_scores)
    cosine_norm = _min_max_norm(cosine_scores)
    if cls_scores is not None:
        cls_norm = _min_max_norm(cls_scores)
        combined = (EVIDENCE_BM25_WEIGHT * bm25_norm + EVIDENCE_COSINE_WEIGHT * cosine_norm
                    + EVIDENCE_CLS_WEIGHT * cls_norm)
    else:
        w_sum = EVIDENCE_BM25_WEIGHT + EVIDENCE_COSINE_WEIGHT
        combined = (EVIDENCE_BM25_WEIGHT / w_sum) * bm25_norm + (EVIDENCE_COSINE_WEIGHT / w_sum) * cosine_norm

    top_idx = np.argsort(-combined)[:TOP_M_EVIDENCE]
    return [sentences[i] for i in sorted(top_idx)]

# =============================================================================
# STEP 10: LLM-BASED REASONING GENERATION (Qwen, Chain-of-Thought prompting)
#   Input to LLM (IPC Section, Selected Evidence Sentences, CoT prompting)
#   -> Qwen -> Output (Predicted IPC sections, Evidence Sentences, Explanation)
# =============================================================================
print("\n" + "=" * 70)
print(f"STEP 10: Loading Qwen ({QWEN_MODEL_NAME}) for reasoning generation")
print("=" * 70)

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE).eval()


def _build_cot_prompt(fact_snippet, section, section_title, evidence_sentences):
    evidence_block = "\n".join(f"- {s}" for s in evidence_sentences) or "(no distinct evidence sentence found)"
    return (
        f"You are a legal reasoning assistant for Indian Penal Code (IPC) section attribution.\n\n"
        f"Case fact (relevant excerpt):\n{fact_snippet}\n\n"
        f"Candidate IPC section: {section} ({section_title})\n"
        f"Evidence sentences retrieved from the case for this section:\n{evidence_block}\n\n"
        f"Think step by step (chain of thought): first restate what {section} legally requires, "
        f"then check whether the evidence sentences above satisfy each requirement, then decide.\n"
        f"Respond with ONLY a JSON object, no extra text, in this exact schema:\n"
        f'{{"applies": true or false, "evidence_sentences": [the evidence sentences you actually relied on], '
        f'"explanation": "one or two sentence justification"}}'
    )


@torch.no_grad()
def qwen_reason_about_candidate(fact_text, section, evidence_sentences):
    section_title = ipc_titles.get(section, "")
    fact_snippet = fact_text[:1500]
    prompt = _build_cot_prompt(fact_snippet, section, section_title, evidence_sentences)
    messages = [{"role": "user", "content": prompt}]
    # apply_chat_template(..., return_tensors="pt") returns a BatchEncoding
    # (dict-like), not a bare tensor, in newer `transformers` -- ask for the
    # dict explicitly and pull input_ids / attention_mask out by name.
    encoded = qwen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(DEVICE)
    input_ids = encoded["input_ids"]
    attention_mask = encoded.get("attention_mask")
    output_ids = qwen_model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        max_new_tokens=LLM_MAX_NEW_TOKENS, do_sample=LLM_TEMPERATURE > 0,
        temperature=max(LLM_TEMPERATURE, 1e-5), pad_token_id=qwen_tokenizer.eos_token_id,
    )
    generated = qwen_tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)

    parsed = {"applies": True, "evidence_sentences": evidence_sentences, "explanation": generated.strip()}
    match = re.search(r"\{.*\}", generated, flags=re.DOTALL)
    if match:
        try:
            candidate = json.loads(match.group(0))
            parsed["applies"] = bool(candidate.get("applies", True))
            parsed["evidence_sentences"] = candidate.get("evidence_sentences", evidence_sentences) or evidence_sentences
            parsed["explanation"] = str(candidate.get("explanation", "")).strip() or generated.strip()
        except Exception:
            pass
    return parsed


def predict_document_hybrid(fact_text):
    """Case Facts -> Classification + Retrieval branches -> Fusion ->
    (FIXED) statute decision -> Evidence Retrieval -> LLM Reasoning ->
    final {section, evidence_sentences, explanation, llm_agrees} records.

    NOTE: `llm_agrees` (the LLM's own applies/does-not-apply verdict) is
    recorded for inspection but does NOT remove a section from the official
    predicted set -- the predicted set itself is exactly `predict_statutes`,
    the already-fixed decision rule. If you want the LLM allowed to veto a
    prediction, filter on `llm_agrees` yourself after Step 11 and re-run
    compute_classification_metrics on the filtered set -- do this as a
    separate experiment so you can compare Macro-F1 with/without the veto
    rather than silently changing the reported number.
    """
    cls_probs, alpha, offsets, mask = classifier_forward_for(fact_text)
    retr_scores = retrieval_scores_for(fact_text)
    predicted_sections = predict_statutes(cls_probs, retr_scores)

    results = []
    for section in predicted_sections:
        evidence = evidence_sentences_for_candidate(fact_text, section, alpha, offsets, mask)
        llm_out = qwen_reason_about_candidate(fact_text, section, evidence)
        results.append({
            "section": section,
            "evidence_sentences": llm_out["evidence_sentences"] or evidence,
            "explanation": llm_out["explanation"],
            "llm_agrees": llm_out["applies"],
        })
    return results

# =============================================================================
# STEP 11: RUN ON THE HELD-OUT TEST SET
# =============================================================================
print("\n" + "=" * 70)
print(f"STEP 11: Predicting on {len(test_docs)} held-out TEST documents")
print("=" * 70)

all_predictions = {}
for d in test_docs:
    statute_preds = predict_document_hybrid(d["fact"])
    all_predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": statute_preds}
    pred_secs = [p["section"] for p in statute_preds]
    print(f"{d['doc_id']} -> pred={pred_secs} | gold={gold_sections_of(d)}")

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for rec in all_predictions.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"\nPredictions saved to: {PREDICTIONS_PATH}")

# =============================================================================
# STEP 12: EVALUATION -- Macro-F1 / Micro-F1 / Accuracy (classification) +
#          ROUGE-L / BLEU / METEOR (generated explanation quality) + Total
# =============================================================================
print("\n" + "=" * 70)
print("STEP 12: Evaluation against gold labels")
print("=" * 70)


def pred_sections_of(pred_rec):
    return [p["section"] for p in pred_rec.get("statute", [])]


def pred_explanation_of(pred_rec):
    return " ".join(p.get("explanation", "") for p in pred_rec.get("statute", []))


rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smoothing = SmoothingFunction().method1


def compute_classification_metrics(test_docs, predictions):
    gold_labels = [gold_sections_of(d) for d in test_docs]
    pred_labels = [pred_sections_of(predictions[d["doc_id"]]) for d in test_docs]
    all_labels = sorted(set(l for labels in (gold_labels + pred_labels) for l in labels))
    mlb = MultiLabelBinarizer(classes=all_labels)
    y_true = mlb.fit_transform(gold_labels)
    y_pred = mlb.transform(pred_labels)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    exact_matches = sum(
        1 for d in test_docs
        if set(pred_sections_of(predictions[d["doc_id"]])) == set(gold_sections_of(d))
    )
    accuracy = exact_matches / len(test_docs)
    return macro_f1, micro_f1, accuracy


def compute_generation_metrics(test_docs, predictions):
    rouge_l_scores, bleu_scores, meteor_scores = [], [], []
    for d in test_docs:
        reference = gold_explanation_reference(d)
        hypothesis = pred_explanation_of(predictions[d["doc_id"]])
        if not hypothesis.strip():
            continue
        rouge_l_scores.append(rouge.score(reference, hypothesis)["rougeL"].fmeasure)
        ref_tokens, hyp_tokens = reference.split(), hypothesis.split()
        bleu_scores.append(sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing))
        try:
            meteor_scores.append(meteor_score([ref_tokens], hyp_tokens))
        except Exception:
            pass
    rouge_l = float(np.mean(rouge_l_scores)) if rouge_l_scores else 0.0
    bleu = float(np.mean(bleu_scores)) if bleu_scores else 0.0
    meteor = float(np.mean(meteor_scores)) if meteor_scores else 0.0
    return rouge_l, bleu, meteor


macro_f1, micro_f1, accuracy = compute_classification_metrics(test_docs, all_predictions)
rouge_l, bleu, meteor = compute_generation_metrics(test_docs, all_predictions)
metric_values = {
    "Macro-F1": macro_f1, "Micro-F1": micro_f1, "Accuracy": accuracy,
    "ROUGE-L": rouge_l, "BLEU": bleu, "METEOR": meteor,
}
total = float(np.mean(list(metric_values.values())))

print("\nPerformance report")
print("  ".join(f"{k:>10s}" for k in metric_values) + f"  {'Total':>10s}")
print("  ".join(f"{metric_values[k]:>10.4f}" for k in metric_values) + f"  {total:>10.4f}")

# =============================================================================
# STEP 13: SIDE-BY-SIDE COMPARISON CSV
# =============================================================================
print("\n" + "=" * 70)
print("STEP 13: Writing predicted-vs-gold comparison CSV")
print("=" * 70)


def write_comparison_csv(test_docs, predictions, path):
    import csv
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "doc_id", "fact_snippet", "gold_sections", "predicted_sections",
            "correct_sections", "missed_sections", "extra_sections", "exact_match",
            "evidence_sentences", "explanation",
        ])
        for d in test_docs:
            gold = set(gold_sections_of(d))
            rec = predictions[d["doc_id"]]
            pred = set(pred_sections_of(rec))
            fact_snippet = (d.get("fact", "") or "")[:150].replace("\n", " ")
            evidence_all = "; ".join(s for p in rec["statute"] for s in p.get("evidence_sentences", []))
            writer.writerow([
                d["doc_id"], fact_snippet,
                "; ".join(sorted(gold)), "; ".join(sorted(pred)),
                "; ".join(sorted(gold & pred)), "; ".join(sorted(gold - pred)), "; ".join(sorted(pred - gold)),
                "YES" if gold == pred else "NO",
                evidence_all[:500], pred_explanation_of(rec)[:500],
            ])
    print(f"Wrote predicted-vs-gold comparison to: {path}")


write_comparison_csv(test_docs, all_predictions, COMPARISON_PATH)
print("\nDONE.")

Torch: 2.14.0+cu130 | CUDA available: True
Device: cuda
STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits
Loaded 575 official IPC sections from ipc_sections_clean.json
Total docs: 525 | Train: 378 | Val: 42 | Test: 105

STEP 4: Building the hybrid label space (classifier vs retrieval-only)
7 distinct gold sections seen in TRAIN docs.
-> 7 kept as CLASSIFIER classes (frequency >= 3).
-> remaining 568 of 575 sections are RETRIEVAL-ONLY.

STEP 5: Supervised Classification Branch (InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[classifier] epoch 1/8 -- avg loss: 1.9234
[classifier] epoch 2/8 -- avg loss: 1.6342
[classifier] epoch 3/8 -- avg loss: 1.2731
[classifier] epoch 4/8 -- avg loss: 1.0698
[classifier] epoch 5/8 -- avg loss: 0.9210
[classifier] epoch 6/8 -- avg loss: 0.8153
[classifier] epoch 7/8 -- avg loss: 0.7345
[classifier] epoch 8/8 -- avg loss: 0.6888
Classifier saved to ./classifier_out

STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building BM25 index over the IPC Knowledge Base ...
Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...
Catalog embedding matrix: (575, 768)

STEP 7: Calibrating per-class classifier thresholds on VAL
Calibrated 7 per-class thresholds: {'IPC 147': 0.18, 'IPC 201': 0.12000000000000001, 'IPC 302': 0.22, 'IPC 376': 0.13999999999999999, 'IPC 420': 0.13999999999999999, 'IPC 498A': 0.28, 'IPC 506': 0.06}

STEP 8: Fusion -> thresholded multi-label statute prediction

STEP 9: Evidence Sentence Retrieval for each predicted IPC

STEP 10: Loading Qwen (Qwen/Qwen2.5-1.5B-Instruct) for reasoning generation


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


STEP 11: Predicting on 105 held-out TEST documents
2002.INSC.274.txt -> pred=['IPC 376'] | gold=['IPC 376']
1999.INSC.378.txt -> pred=['IPC 376', 'IPC 498A', 'IPC 506'] | gold=['IPC 201', 'IPC 302']
2012.INSC.512.txt -> pred=['IPC 201', 'IPC 302'] | gold=['IPC 498A']
1998.INSC.126.txt -> pred=['IPC 302'] | gold=['IPC 302']
2007.INSC.590.txt -> pred=['IPC 147', 'IPC 302', 'IPC 506'] | gold=['IPC 147']
2013.INSC.960.txt -> pred=['IPC 420', 'IPC 506'] | gold=['IPC 201']
2009.INSC.1130.txt -> pred=['IPC 376'] | gold=['IPC 376']
1999.INSC.175.txt -> pred=['IPC 147', 'IPC 302', 'IPC 506'] | gold=['IPC 302']
2003.INSC.597.txt -> pred=['IPC 147', 'IPC 302'] | gold=['IPC 302']
1998.INSC.474.txt -> pred=['IPC 147', 'IPC 302', 'IPC 506'] | gold=['IPC 302']
2015.INSC.345.txt -> pred=['IPC 147', 'IPC 302', 'IPC 506'] | gold=['IPC 302', 'IPC 506']
2011.INSC.316.txt -> pred=['IPC 147', 'IPC 302', 'IPC 506'] | gold=['IPC 147', 'IPC 302']
2016.INSC.429.txt -> pred=['IPC 201', 'IPC 498A'] | gold=['IPC 

TypeError: sequence item 0: expected str instance, dict found

In [6]:
"""
ipc_hybrid_full_pipeline_fixed.py
==================================
Full "Classify-Retrieve-Evidence-Reason" pipeline, built ON TOP OF the
CORRECTED statute-decision logic from ipc_hybrid_statute_only.py (the
version that fixed Macro-F1). This restores the two boxes that were
removed for speed -- Evidence Sentence Retrieval and LLM-Based Reasoning
Generation -- without touching the fixed decision logic that got Macro-F1
back to a sane value.

    Case Facts -> [Supervised Classification Branch] + [IPC Retrieval Branch]
               -> Score Normalization and Fusion -> IPC Candidate Ranking
               -> Evidence Sentence Retrieval for each candidate IPC
               -> LLM-Based Reasoning Generation (Qwen, CoT)
               -> Predicted IPC sections + Evidence Sentences + Explanation

IMPORTANT DESIGN DECISION (read this before changing anything):
-----------------------------------------------------------------
Two bugs already cost you a lot of Macro-F1 in earlier iterations:
  1. Always emitting the top-K fused candidates as the final prediction
     (no filtering) -> Macro-F1 0.14 (spam from retrieval-only sections).
  2. Comparing a PER-DOCUMENT min-max-normalized retrieval score against a
     FIXED global threshold -> Macro-F1 0.03 (even worse spam).
The fix that got you back to a sane Macro-F1 was: predicted sections come
ONLY from the classifier branch's calibrated per-class thresholds (with a
classifier-only single-best fallback so nothing is ever empty). That
decision rule is UNCHANGED here -- `predict_statutes()` is copied verbatim.

The "IPC Candidate Ranking" fed into Evidence Retrieval + LLM Reasoning in
this version is therefore simply the ALREADY-DECIDED predicted section set
(normally 1, occasionally 2-3 sections), not a blind top-5. The LLM is used
to explain *why* each already-decided section applies (evidence sentences +
a CoT explanation) -- its own "applies" verdict is recorded for your
inspection but does NOT remove a section from the official prediction,
so it cannot re-introduce the spam bug or silently change your Macro-F1.
If you want the LLM to be allowed to veto a prediction, see the comment
right above `predicted_with_reasoning` in Step 11.

--------------------------------------------------------------------------
BUGFIX (this version): TypeError in write_comparison_csv (Step 13)
--------------------------------------------------------------------------
  TypeError: sequence item 0: expected str instance, dict found
  at: evidence_all = "; ".join(s for p in rec["statute"] for s in p.get("evidence_sentences", []))

Root cause: qwen_reason_about_candidate() parsed the LLM's JSON output and
did `parsed["evidence_sentences"] = candidate.get("evidence_sentences", ...)`
with NO type checking. The CoT prompt asks for "the evidence sentences you
actually relied on" as a JSON array, but nothing stops Qwen from emitting an
array of OBJECTS (e.g. `{"sentence": "...", "note": "..."}`) instead of an
array of plain strings -- the schema in the prompt is advisory, not
enforced. That list-of-dicts then flowed straight through
predict_document_hybrid() into `results[...]["evidence_sentences"]` and
crashed the very first `str.join()` that touched it, in Step 13.

Fix: `_coerce_evidence_list()` below sanitizes whatever the LLM returns for
`evidence_sentences` into a clean list of plain strings -- unpacking common
`{"sentence": ...}` / `{"text": ...}` shapes when present, dropping
anything else it can't turn into text, and falling back to the
retrieval-based evidence sentences (which are always plain strings) if the
LLM's list turns out to contain nothing usable. `write_comparison_csv` also
wraps every value in `str(...)` as a second line of defense, so a future
malformed field degrades gracefully into readable text in the CSV instead
of crashing the whole run.

Expected input files (unchanged):
  - task1.jsonl            : one JSON object per line, each with at least
                              {"doc_id": ..., "fact": ..., "statute": [...],
                               "explanation": {sentence: ipc_label, ...}}
  - ipc_sections_clean.json: [{"section": "302", "title": ..., "text": ...}, ...]
"""

# =============================================================================
# STEP 0: DEPENDENCIES
# =============================================================================
import subprocess
import sys


def ensure_packages():
    import importlib
    pkgs = {
        "torch": "torch",
        "transformers": "transformers",
        "scikit-learn": "sklearn",
        "accelerate": "accelerate",
        "numpy": "numpy",
        "rank_bm25": "rank_bm25",
        "rouge_score": "rouge_score",
        "nltk": "nltk",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing '{pip_name}' ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)

    import nltk
    for res, pkg in [("tokenizers/punkt", "punkt"), ("tokenizers/punkt_tab", "punkt_tab"),
                      ("corpora/wordnet", "wordnet"), ("corpora/omw-1.4", "omw-1.4")]:
        try:
            nltk.data.find(res)
        except LookupError:
            try:
                nltk.download(pkg, quiet=True)
            except Exception:
                pass


ensure_packages()

import json
import os
import re
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForCausalLM, get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from rank_bm25 import BM25Okapi
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# =============================================================================
# STEP 1: CONFIG
# =============================================================================
TASK1_PATH = "task1.jsonl"
IPC_KB_PATH = "ipc_sections_clean.json"
OUTPUT_DIR_CLASSIFIER = "./classifier_out"
PREDICTIONS_PATH = "predictions_full_pipeline.jsonl"
COMPARISON_PATH = "comparison_pred_vs_gold.csv"

RANDOM_SEED = 42
TEST_FRACTION = 0.20
VAL_FRACTION = 0.10

MODEL_NAME = "law-ai/InLegalBERT"

# --- Supervised Classification Branch ---
MIN_CLASSIFIER_LABEL_FREQ = 3
CLASSIFIER_MAX_LENGTH = 384
CLASSIFIER_BATCH_SIZE = 8
CLASSIFIER_EPOCHS = 8
CLASSIFIER_LR = 2e-5
BILSTM_HIDDEN = 256
ATTN_DIM = 200

# --- IPC Retrieval Branch ---
RETRIEVAL_MAX_TOKEN_LEN = 256
RETRIEVAL_BM25_WEIGHT = 0.5
RETRIEVAL_COSINE_WEIGHT = 0.5

# --- Calibration (this is what actually decides the predicted sections) ---
THRESHOLD_SEARCH_MIN = 0.02
THRESHOLD_SEARCH_MAX = 0.60
THRESHOLD_SEARCH_STEP = 0.02
DEFAULT_CLASSIFIER_THRESHOLD = 0.15

# --- Retrieval-only sections: OFF by default -- see the long comment in
#     Step 8 (predict_statutes) for why. Left here only for completeness. ---
ENABLE_RETRIEVAL_ONLY_PREDICTIONS = False
RETRIEVAL_ONLY_THRESHOLD = 0.85  # unused while the flag above is False

# --- Evidence Sentence Retrieval for Each Candidate IPC ---
EVIDENCE_BM25_WEIGHT = 0.34
EVIDENCE_COSINE_WEIGHT = 0.33
EVIDENCE_CLS_WEIGHT = 0.33
TOP_M_EVIDENCE = 3

# --- LLM-Based Reasoning Generation (Qwen, CoT prompting) ---
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MAX_NEW_TOKENS = 300
LLM_TEMPERATURE = 0.2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# =============================================================================
# STEP 2: SHARED TEXT UTILITIES
# =============================================================================
_ABBREV_PATTERNS = [
    r"\bPW-?\d*\.", r"\bp\.m\.", r"\ba\.m\.", r"\bExt\.-?", r"\bRs\.",
    r"\bNo\.", r"\bSec\.", r"\bSection\.", r"\bvs\.", r"\bv\.", r"\bMr\.",
    r"\bMrs\.", r"\bDr\.", r"\bJ\.\)", r"\bi\.e\.", r"\be\.g\.", r"\bIPC\.",
    r"\bCrPC\.", r"\bHon'ble\.", r"\bU/s\.",
]
_PLACEHOLDER = "<<DOT_{}>>"


def split_sentences_with_spans(text):
    protected = text
    placeholders = {}
    for i, pat in enumerate(_ABBREV_PATTERNS):
        def _sub(m, i=i):
            key = _PLACEHOLDER.format(f"{i}_{len(placeholders)}")
            placeholders[key] = m.group(0)
            return key
        protected = re.sub(pat, _sub, protected)

    raw_sents = re.split(r"(?<=[.!?])\s+(?=[A-Z(\"\u2018\u201c])", protected)

    results = []
    cursor = 0
    for s in raw_sents:
        for key, val in placeholders.items():
            s = s.replace(key, val)
        s_stripped = s.strip()
        if not s_stripped:
            continue
        idx = text.find(s_stripped, cursor)
        if idx == -1:
            idx = text.find(s_stripped)
        if idx == -1:
            start, end = cursor, cursor + len(s_stripped)
        else:
            start, end = idx, idx + len(s_stripped)
        results.append((s_stripped, start, end))
        cursor = end
    return results


def split_sentences(text):
    return [s for s, _, _ in split_sentences_with_spans(text)]


def normalize_ipc_label(label):
    label = str(label).strip()
    m = re.search(r"(\d+[A-Za-z]*)", label)
    if not m:
        return None
    return f"IPC {m.group(1).upper()}"


def load_jsonl_ordered(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def load_ipc_catalog(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    catalog, titles = {}, {}
    for entry in raw:
        code = str(entry.get("section", "")).strip()
        if not code:
            continue
        title = str(entry.get("title", "")).strip()
        text = str(entry.get("text", "")).strip()
        catalog[code] = f"{title}. {text}" if title else text
        titles[code] = title
    return catalog, titles, sorted(catalog.keys())


def gold_sections_of(doc):
    return sorted({s for s in (normalize_ipc_label(g) for g in doc.get("statute", [])) if s})


def gold_explanation_reference(doc):
    """Reference text for ROUGE/BLEU/METEOR: gold (sentence -> IPC) pairs
    joined into one paragraph -- there is no other free-text gold
    'reasoning' available in task1.jsonl."""
    exp = doc.get("explanation", {}) or {}
    if not exp:
        return " ".join(split_sentences(doc.get("fact", ""))[:2])
    return " ".join(f"{sent.strip()} (=> {label})" for sent, label in exp.items())


def _bm25_tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def _min_max_norm(arr):
    arr = np.asarray(arr, dtype=np.float64)
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-9:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

# =============================================================================
# STEP 3: LOAD DATA + SPLIT INTO TRAIN / VAL / TEST
# =============================================================================
print("=" * 70)
print("STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits")
print("=" * 70)

ipc_catalog, ipc_titles, all_section_codes_raw = load_ipc_catalog(IPC_KB_PATH)
print(f"Loaded {len(all_section_codes_raw)} official IPC sections from {IPC_KB_PATH}")

norm_to_raw = {normalize_ipc_label(c): c for c in all_section_codes_raw}
all_section_codes = sorted(norm_to_raw.keys())
section_index = {sec: i for i, sec in enumerate(all_section_codes)}
n_sections = len(all_section_codes)

all_docs = load_jsonl_ordered(TASK1_PATH)
random.Random(RANDOM_SEED).shuffle(all_docs)

n = len(all_docs)
n_test = max(1, int(n * TEST_FRACTION))
n_val = max(1, int((n - n_test) * VAL_FRACTION))
test_docs = all_docs[:n_test]
val_docs = all_docs[n_test:n_test + n_val]
train_docs = all_docs[n_test + n_val:]
print(f"Total docs: {n} | Train: {len(train_docs)} | Val: {len(val_docs)} | Test: {len(test_docs)}")

# =============================================================================
# STEP 4: HYBRID LABEL SPACE (7 classifier classes vs retrieval-only)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: Building the hybrid label space (classifier vs retrieval-only)")
print("=" * 70)

doc_label_counts = Counter()
for d in train_docs:
    for s in gold_sections_of(d):
        doc_label_counts[s] += 1

classifier_label_list = sorted([s for s, c in doc_label_counts.items() if c >= MIN_CLASSIFIER_LABEL_FREQ])
label_to_idx = {s: i for i, s in enumerate(classifier_label_list)}
idx_to_label = {i: s for s, i in label_to_idx.items()}
num_classifier_labels = len(classifier_label_list)

print(f"{len(doc_label_counts)} distinct gold sections seen in TRAIN docs.")
print(f"-> {num_classifier_labels} kept as CLASSIFIER classes (frequency >= {MIN_CLASSIFIER_LABEL_FREQ}).")
print(f"-> remaining {n_sections - num_classifier_labels} of {n_sections} sections are RETRIEVAL-ONLY.")

# =============================================================================
# STEP 5: SUPERVISED CLASSIFICATION BRANCH
#   InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax
#   -> 7 IPC probabilities
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: Supervised Classification Branch "
      "(InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)")
print("=" * 70)

classifier_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_encoder = AutoModel.from_pretrained(MODEL_NAME)
bert_hidden_size = bert_encoder.config.hidden_size


class LabelWiseAttentionClassifier(nn.Module):
    """InLegalBERT -> BiLSTM -> label-wise attention -> Linear & Softmax.
    Returns `alpha` (per-label attention over tokens) too, so the Evidence
    Sentence Retrieval stage can reuse it as 'Classifier-Based Relevance'."""

    def __init__(self, encoder, hidden_size, num_labels, lstm_hidden=BILSTM_HIDDEN, attn_dim=ATTN_DIM):
        super().__init__()
        self.encoder = encoder
        self.bilstm = nn.LSTM(hidden_size, lstm_hidden, batch_first=True, bidirectional=True)
        lstm_out_dim = lstm_hidden * 2
        self.attn_W = nn.Linear(lstm_out_dim, attn_dim, bias=False)
        self.attn_U = nn.Linear(attn_dim, num_labels, bias=False)
        self.label_weight = nn.Parameter(torch.randn(num_labels, lstm_out_dim) * 0.01)
        self.label_bias = nn.Parameter(torch.zeros(num_labels))
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.bilstm(enc_out)
        u = torch.tanh(self.attn_W(lstm_out))
        scores = self.attn_U(u)
        pad_mask = (~attention_mask.bool()).unsqueeze(-1)
        scores = scores.masked_fill(pad_mask, float("-inf"))
        alpha = torch.softmax(scores, dim=1)
        context = torch.einsum("btl,bth->blh", alpha, lstm_out)
        logits = torch.einsum("blh,lh->bl", context, self.label_weight) + self.label_bias
        probs = torch.softmax(logits, dim=-1)
        return logits, probs, alpha


classifier_model = LabelWiseAttentionClassifier(bert_encoder, bert_hidden_size, num_classifier_labels).to(DEVICE)


class StatuteDataset(Dataset):
    def __init__(self, records, label_to_idx, tokenizer, max_length):
        self.records = records
        self.label_to_idx = label_to_idx
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(rec["fact"], truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        target = torch.zeros(len(self.label_to_idx))
        for s in gold_sections_of(rec):
            if s in self.label_to_idx:
                target[self.label_to_idx[s]] = 1.0
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": target,
        }


def soft_target_cross_entropy(logits, multi_hot_targets):
    row_sums = multi_hot_targets.sum(dim=-1, keepdim=True)
    safe_targets = torch.where(row_sums > 0, multi_hot_targets / row_sums.clamp(min=1e-9),
                                torch.full_like(multi_hot_targets, 1.0 / multi_hot_targets.size(-1)))
    log_probs = F.log_softmax(logits, dim=-1)
    return -(safe_targets * log_probs).sum(dim=-1).mean()


train_ds = StatuteDataset(train_docs, label_to_idx, classifier_tokenizer, CLASSIFIER_MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=CLASSIFIER_LR)
total_steps = max(1, len(train_loader) * CLASSIFIER_EPOCHS)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                             num_training_steps=total_steps)

classifier_model.train()
for epoch in range(CLASSIFIER_EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits, probs, _ = classifier_model(input_ids, attention_mask)
        loss = soft_target_cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"[classifier] epoch {epoch + 1}/{CLASSIFIER_EPOCHS} -- avg loss: {total_loss / max(1, len(train_loader)):.4f}")

classifier_model.eval()
os.makedirs(OUTPUT_DIR_CLASSIFIER, exist_ok=True)
torch.save(classifier_model.state_dict(), os.path.join(OUTPUT_DIR_CLASSIFIER, "label_attention_classifier.pt"))
classifier_tokenizer.save_pretrained(OUTPUT_DIR_CLASSIFIER)
print(f"Classifier saved to {OUTPUT_DIR_CLASSIFIER}")


@torch.no_grad()
def classifier_probs_for(fact_text):
    """Lightweight variant (no offsets) -- used for VAL calibration (Step 7)
    where only the 7 probabilities are needed, for speed."""
    enc = classifier_tokenizer(fact_text, truncation=True, padding="max_length",
                                max_length=CLASSIFIER_MAX_LENGTH, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    _, probs, _ = classifier_model(enc["input_ids"], enc["attention_mask"])
    probs = probs.squeeze(0).cpu().numpy()
    return {idx_to_label[i]: float(probs[i]) for i in range(num_classifier_labels)}


@torch.no_grad()
def classifier_forward_for(fact_text):
    """Full variant (with offsets) -- used at test time so Evidence Sentence
    Retrieval can reuse the label-wise attention as 'Classifier-Based
    Relevance'."""
    enc = classifier_tokenizer(fact_text, truncation=True, padding="max_length",
                                max_length=CLASSIFIER_MAX_LENGTH, return_tensors="pt",
                                return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0]
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    _, probs, alpha = classifier_model(enc["input_ids"], enc["attention_mask"])
    probs = probs.squeeze(0).cpu().numpy()
    alpha = alpha.squeeze(0).cpu().numpy()
    attn_mask = enc["attention_mask"].squeeze(0).cpu().numpy()
    probs_by_label = {idx_to_label[i]: float(probs[i]) for i in range(num_classifier_labels)}
    return probs_by_label, alpha, offsets.numpy(), attn_mask


def classifier_attention_per_sentence(fact_text, label, alpha, offsets, attn_mask):
    """Aggregates the label-wise attention mass onto each sentence's
    character span -- this IS 'Classifier-Based Relevance' in the diagram."""
    if label not in label_to_idx:
        return None
    lab_idx = label_to_idx[label]
    sent_spans = split_sentences_with_spans(fact_text)
    if not sent_spans:
        return []
    scores = [0.0] * len(sent_spans)
    for t in range(len(offsets)):
        if attn_mask[t] == 0:
            continue
        tok_start, tok_end = int(offsets[t][0]), int(offsets[t][1])
        if tok_end <= tok_start:
            continue
        for si, (_, s_start, s_end) in enumerate(sent_spans):
            if tok_start < s_end and tok_end > s_start:
                scores[si] += float(alpha[t, lab_idx])
                break
    return scores

# =============================================================================
# STEP 6: IPC RETRIEVAL BRANCH
#   IPC Knowledge Base -> BM25 + Cosine Similarity -> Retrieval Score
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)")
print("=" * 70)

retrieval_tokenizer = classifier_tokenizer
retrieval_encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

print("Building BM25 index over the IPC Knowledge Base ...")
catalog_texts = [ipc_catalog[norm_to_raw[sec]] for sec in all_section_codes]
bm25_index = BM25Okapi([_bm25_tokenize(t) for t in catalog_texts])


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def encode_texts(texts, max_len, batch_size=32):
    embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = retrieval_tokenizer(batch, truncation=True, padding=True, max_length=max_len,
                                   return_tensors="pt").to(DEVICE)
        out = retrieval_encoder(**enc).last_hidden_state
        pooled = F.normalize(mean_pool(out, enc["attention_mask"]), p=2, dim=-1)
        embeds.append(pooled.cpu())
    return torch.cat(embeds, dim=0) if embeds else torch.zeros((0, bert_hidden_size))


print("Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...")
catalog_embeddings = encode_texts(catalog_texts, RETRIEVAL_MAX_TOKEN_LEN)
print(f"Catalog embedding matrix: {tuple(catalog_embeddings.shape)}")


def retrieval_scores_for(fact_text):
    """BM25 + Cosine Similarity -> fused Retrieval Score for all 575 IPC
    sections (PER-DOCUMENT min-max normalized -- fine for ranking WITHIN one
    document, not for comparing across documents against a fixed bar; see
    the Step 8 comment for why that distinction matters)."""
    fact_emb = encode_texts([fact_text], RETRIEVAL_MAX_TOKEN_LEN)
    cosine_scores = (fact_emb @ catalog_embeddings.t()).squeeze(0).numpy()
    bm25_scores = bm25_index.get_scores(_bm25_tokenize(fact_text))

    bm25_norm = _min_max_norm(bm25_scores)
    cosine_norm = _min_max_norm(cosine_scores)
    fused = RETRIEVAL_BM25_WEIGHT * bm25_norm + RETRIEVAL_COSINE_WEIGHT * cosine_norm
    return {all_section_codes[j]: float(fused[j]) for j in range(n_sections)}

# =============================================================================
# STEP 7: CALIBRATION ON THE VALIDATION SET
# =============================================================================
print("\n" + "=" * 70)
print("STEP 7: Calibrating per-class classifier thresholds on VAL")
print("=" * 70)

val_gold = {d["doc_id"]: gold_sections_of(d) for d in val_docs}
val_probs_matrix = np.zeros((len(val_docs), num_classifier_labels))
for i, d in enumerate(val_docs):
    probs = classifier_probs_for(d["fact"])
    for j, lab in enumerate(classifier_label_list):
        val_probs_matrix[i, j] = probs[lab]

val_targets_matrix = np.zeros_like(val_probs_matrix)
for i, d in enumerate(val_docs):
    gold = set(val_gold[d["doc_id"]])
    for j, lab in enumerate(classifier_label_list):
        val_targets_matrix[i, j] = 1.0 if lab in gold else 0.0

classifier_thresholds = {}
grid = np.arange(THRESHOLD_SEARCH_MIN, THRESHOLD_SEARCH_MAX + 1e-9, THRESHOLD_SEARCH_STEP)
for j, lab in enumerate(classifier_label_list):
    y_true = val_targets_matrix[:, j]
    if y_true.sum() == 0:
        classifier_thresholds[lab] = DEFAULT_CLASSIFIER_THRESHOLD
        continue
    best_t, best_f1 = DEFAULT_CLASSIFIER_THRESHOLD, -1.0
    for t in grid:
        y_pred = (val_probs_matrix[:, j] >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    classifier_thresholds[lab] = best_t
print(f"Calibrated {len(classifier_thresholds)} per-class thresholds: {classifier_thresholds}")

# =============================================================================
# STEP 8: SCORE NORMALIZATION AND FUSION -> IPC CANDIDATE RANKING
#   (this decision rule is UNCHANGED from the fixed statute-only version --
#   do not blindly take top-K here, that is the bug that broke Macro-F1
#   twice already)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 8: Fusion -> thresholded multi-label statute prediction")
print("=" * 70)


def predict_statutes(cls_probs, retr_scores):
    predicted = set()

    # (a) classifier branch: multi-label decision via each class's OWN
    #     calibrated threshold -- the actual fix for Macro-F1.
    for lab, p in cls_probs.items():
        if p >= classifier_thresholds[lab]:
            predicted.add(lab)

    # (b) retrieval-only sections: OFF by default. retr_scores is min-max
    #     normalized PER DOCUMENT, so comparing it to a fixed global bar
    #     across documents reintroduces the spam bug that gave Macro-F1 =
    #     0.03 last time. Only flip this on after re-deriving a globally
    #     comparable retrieval score and recalibrating on VAL like Step 7.
    if ENABLE_RETRIEVAL_ONLY_PREDICTIONS:
        for sec, score in retr_scores.items():
            if sec not in label_to_idx and score >= RETRIEVAL_ONLY_THRESHOLD:
                predicted.add(sec)

    # (c) never emit an empty prediction: fall back to the classifier's own
    #     single best class (not a retrieval-influenced fused score).
    if not predicted:
        predicted.add(max(cls_probs, key=cls_probs.get))

    return sorted(predicted)

# =============================================================================
# STEP 9: EVIDENCE SENTENCE RETRIEVAL FOR EACH CANDIDATE IPC
#   Sentence Scoring S = {S1..Sn} via BM25 + Cosine Similarity +
#   Classifier-Based Relevance -> Top-m Evidence Sentences
# =============================================================================
print("\n" + "=" * 70)
print("STEP 9: Evidence Sentence Retrieval for each predicted IPC")
print("=" * 70)


def evidence_sentences_for_candidate(fact_text, section, alpha, offsets, mask):
    sentences = split_sentences(fact_text)
    if not sentences:
        return []
    section_text = ipc_catalog[norm_to_raw[section]]

    sent_tokens = [_bm25_tokenize(s) for s in sentences]
    local_bm25 = BM25Okapi(sent_tokens) if sent_tokens else None
    bm25_scores = local_bm25.get_scores(_bm25_tokenize(section_text)) if local_bm25 else np.zeros(len(sentences))

    sent_embs = encode_texts(sentences, RETRIEVAL_MAX_TOKEN_LEN)
    section_emb = encode_texts([section_text], RETRIEVAL_MAX_TOKEN_LEN)
    cosine_scores = (sent_embs @ section_emb.t()).squeeze(-1).numpy()

    cls_scores = classifier_attention_per_sentence(fact_text, section, alpha, offsets, mask)

    bm25_norm = _min_max_norm(bm25_scores)
    cosine_norm = _min_max_norm(cosine_scores)
    if cls_scores is not None:
        cls_norm = _min_max_norm(cls_scores)
        combined = (EVIDENCE_BM25_WEIGHT * bm25_norm + EVIDENCE_COSINE_WEIGHT * cosine_norm
                    + EVIDENCE_CLS_WEIGHT * cls_norm)
    else:
        w_sum = EVIDENCE_BM25_WEIGHT + EVIDENCE_COSINE_WEIGHT
        combined = (EVIDENCE_BM25_WEIGHT / w_sum) * bm25_norm + (EVIDENCE_COSINE_WEIGHT / w_sum) * cosine_norm

    top_idx = np.argsort(-combined)[:TOP_M_EVIDENCE]
    return [sentences[i] for i in sorted(top_idx)]

# =============================================================================
# STEP 10: LLM-BASED REASONING GENERATION (Qwen, Chain-of-Thought prompting)
#   Input to LLM (IPC Section, Selected Evidence Sentences, CoT prompting)
#   -> Qwen -> Output (Predicted IPC sections, Evidence Sentences, Explanation)
# =============================================================================
print("\n" + "=" * 70)
print(f"STEP 10: Loading Qwen ({QWEN_MODEL_NAME}) for reasoning generation")
print("=" * 70)

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE).eval()


def _build_cot_prompt(fact_snippet, section, section_title, evidence_sentences):
    evidence_block = "\n".join(f"- {s}" for s in evidence_sentences) or "(no distinct evidence sentence found)"
    return (
        f"You are a legal reasoning assistant for Indian Penal Code (IPC) section attribution.\n\n"
        f"Case fact (relevant excerpt):\n{fact_snippet}\n\n"
        f"Candidate IPC section: {section} ({section_title})\n"
        f"Evidence sentences retrieved from the case for this section:\n{evidence_block}\n\n"
        f"Think step by step (chain of thought): first restate what {section} legally requires, "
        f"then check whether the evidence sentences above satisfy each requirement, then decide.\n"
        f"Respond with ONLY a JSON object, no extra text, in this exact schema:\n"
        f'{{"applies": true or false, "evidence_sentences": [the evidence sentences you actually relied on, '
        f'as PLAIN STRINGS -- do not wrap them in objects], '
        f'"explanation": "one or two sentence justification"}}'
    )


def _coerce_evidence_list(value, fallback):
    """Sanitizes whatever the LLM's JSON put in "evidence_sentences" into a
    clean list of plain strings. The CoT prompt asks for plain strings, but
    the schema is advisory, not enforced -- Qwen can (and sometimes does)
    return a list of objects like {"sentence": "...", ...} instead. This is
    the fix for the `TypeError: sequence item 0: expected str instance,
    dict found` crash in write_comparison_csv: nothing downstream of this
    function should ever see a non-string evidence item again."""
    if not isinstance(value, list):
        return list(fallback)
    out = []
    for item in value:
        if isinstance(item, str):
            s = item.strip()
            if s:
                out.append(s)
        elif isinstance(item, dict):
            # common alternate shapes the model might emit
            for key in ("sentence", "text", "evidence", "evidence_sentence", "content"):
                cand = item.get(key)
                if isinstance(cand, str) and cand.strip():
                    out.append(cand.strip())
                    break
        # anything else (numbers, None, nested lists, ...) is silently dropped
    return out if out else list(fallback)


@torch.no_grad()
def qwen_reason_about_candidate(fact_text, section, evidence_sentences):
    section_title = ipc_titles.get(section, "")
    fact_snippet = fact_text[:1500]
    prompt = _build_cot_prompt(fact_snippet, section, section_title, evidence_sentences)
    messages = [{"role": "user", "content": prompt}]
    # apply_chat_template(..., return_tensors="pt") returns a BatchEncoding
    # (dict-like), not a bare tensor, in newer `transformers` -- ask for the
    # dict explicitly and pull input_ids / attention_mask out by name.
    encoded = qwen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(DEVICE)
    input_ids = encoded["input_ids"]
    attention_mask = encoded.get("attention_mask")
    output_ids = qwen_model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        max_new_tokens=LLM_MAX_NEW_TOKENS, do_sample=LLM_TEMPERATURE > 0,
        temperature=max(LLM_TEMPERATURE, 1e-5), pad_token_id=qwen_tokenizer.eos_token_id,
    )
    generated = qwen_tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)

    # Defaults: if parsing fails entirely, fall back to the retrieval-based
    # evidence (already plain strings) and the raw generated text.
    parsed = {"applies": True, "evidence_sentences": list(evidence_sentences), "explanation": generated.strip()}
    match = re.search(r"\{.*\}", generated, flags=re.DOTALL)
    if match:
        try:
            candidate = json.loads(match.group(0))
            parsed["applies"] = bool(candidate.get("applies", True))
            # THE FIX: sanitize into plain strings instead of trusting the
            # LLM's JSON shape directly.
            parsed["evidence_sentences"] = _coerce_evidence_list(
                candidate.get("evidence_sentences", evidence_sentences), evidence_sentences)
            parsed["explanation"] = str(candidate.get("explanation", "")).strip() or generated.strip()
        except Exception:
            pass
    return parsed


def predict_document_hybrid(fact_text):
    """Case Facts -> Classification + Retrieval branches -> Fusion ->
    (FIXED) statute decision -> Evidence Retrieval -> LLM Reasoning ->
    final {section, evidence_sentences, explanation, llm_agrees} records.

    NOTE: `llm_agrees` (the LLM's own applies/does-not-apply verdict) is
    recorded for inspection but does NOT remove a section from the official
    predicted set -- the predicted set itself is exactly `predict_statutes`,
    the already-fixed decision rule. If you want the LLM allowed to veto a
    prediction, filter on `llm_agrees` yourself after Step 11 and re-run
    compute_classification_metrics on the filtered set -- do this as a
    separate experiment so you can compare Macro-F1 with/without the veto
    rather than silently changing the reported number.
    """
    cls_probs, alpha, offsets, mask = classifier_forward_for(fact_text)
    retr_scores = retrieval_scores_for(fact_text)
    predicted_sections = predict_statutes(cls_probs, retr_scores)

    results = []
    for section in predicted_sections:
        evidence = evidence_sentences_for_candidate(fact_text, section, alpha, offsets, mask)
        llm_out = qwen_reason_about_candidate(fact_text, section, evidence)
        # llm_out["evidence_sentences"] is already sanitized to plain strings
        # by _coerce_evidence_list(); `evidence` (retrieval-based) is the
        # fallback if the LLM's list came back empty after sanitizing.
        results.append({
            "section": section,
            "evidence_sentences": llm_out["evidence_sentences"] or evidence,
            "explanation": llm_out["explanation"],
            "llm_agrees": llm_out["applies"],
        })
    return results

# =============================================================================
# STEP 11: RUN ON THE HELD-OUT TEST SET
# =============================================================================
print("\n" + "=" * 70)
print(f"STEP 11: Predicting on {len(test_docs)} held-out TEST documents")
print("=" * 70)

all_predictions = {}
for d in test_docs:
    statute_preds = predict_document_hybrid(d["fact"])
    all_predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": statute_preds}
    pred_secs = [p["section"] for p in statute_preds]
    print(f"{d['doc_id']} -> pred={pred_secs} | gold={gold_sections_of(d)}")

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for rec in all_predictions.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"\nPredictions saved to: {PREDICTIONS_PATH}")

# =============================================================================
# STEP 12: EVALUATION -- Macro-F1 / Micro-F1 / Accuracy (classification) +
#          ROUGE-L / BLEU / METEOR (generated explanation quality) + Total
# =============================================================================
print("\n" + "=" * 70)
print("STEP 12: Evaluation against gold labels")
print("=" * 70)


def pred_sections_of(pred_rec):
    return [p["section"] for p in pred_rec.get("statute", [])]


def pred_explanation_of(pred_rec):
    return " ".join(p.get("explanation", "") for p in pred_rec.get("statute", []))


rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smoothing = SmoothingFunction().method1


def compute_classification_metrics(test_docs, predictions):
    gold_labels = [gold_sections_of(d) for d in test_docs]
    pred_labels = [pred_sections_of(predictions[d["doc_id"]]) for d in test_docs]
    all_labels = sorted(set(l for labels in (gold_labels + pred_labels) for l in labels))
    mlb = MultiLabelBinarizer(classes=all_labels)
    y_true = mlb.fit_transform(gold_labels)
    y_pred = mlb.transform(pred_labels)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    exact_matches = sum(
        1 for d in test_docs
        if set(pred_sections_of(predictions[d["doc_id"]])) == set(gold_sections_of(d))
    )
    accuracy = exact_matches / len(test_docs)
    return macro_f1, micro_f1, accuracy


def compute_generation_metrics(test_docs, predictions):
    rouge_l_scores, bleu_scores, meteor_scores = [], [], []
    for d in test_docs:
        reference = gold_explanation_reference(d)
        hypothesis = pred_explanation_of(predictions[d["doc_id"]])
        if not hypothesis.strip():
            continue
        rouge_l_scores.append(rouge.score(reference, hypothesis)["rougeL"].fmeasure)
        ref_tokens, hyp_tokens = reference.split(), hypothesis.split()
        bleu_scores.append(sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing))
        try:
            meteor_scores.append(meteor_score([ref_tokens], hyp_tokens))
        except Exception:
            pass
    rouge_l = float(np.mean(rouge_l_scores)) if rouge_l_scores else 0.0
    bleu = float(np.mean(bleu_scores)) if bleu_scores else 0.0
    meteor = float(np.mean(meteor_scores)) if meteor_scores else 0.0
    return rouge_l, bleu, meteor


macro_f1, micro_f1, accuracy = compute_classification_metrics(test_docs, all_predictions)
rouge_l, bleu, meteor = compute_generation_metrics(test_docs, all_predictions)
metric_values = {
    "Macro-F1": macro_f1, "Micro-F1": micro_f1, "Accuracy": accuracy,
    "ROUGE-L": rouge_l, "BLEU": bleu, "METEOR": meteor,
}
total = float(np.mean(list(metric_values.values())))

print("\nPerformance report")
print("  ".join(f"{k:>10s}" for k in metric_values) + f"  {'Total':>10s}")
print("  ".join(f"{metric_values[k]:>10.4f}" for k in metric_values) + f"  {total:>10.4f}")

# =============================================================================
# STEP 13: SIDE-BY-SIDE COMPARISON CSV
# =============================================================================
print("\n" + "=" * 70)
print("STEP 13: Writing predicted-vs-gold comparison CSV")
print("=" * 70)


def write_comparison_csv(test_docs, predictions, path):
    import csv
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "doc_id", "fact_snippet", "gold_sections", "predicted_sections",
            "correct_sections", "missed_sections", "extra_sections", "exact_match",
            "evidence_sentences", "explanation",
        ])
        for d in test_docs:
            gold = set(gold_sections_of(d))
            rec = predictions[d["doc_id"]]
            pred = set(pred_sections_of(rec))
            fact_snippet = (d.get("fact", "") or "")[:150].replace("\n", " ")
            # THE FIX (second line of defense): even after _coerce_evidence_list()
            # sanitizes at the source, wrap every item in str(...) here too, so a
            # future malformed field degrades to readable text instead of
            # crashing str.join() with a TypeError.
            evidence_all = "; ".join(
                str(s) for p in rec["statute"] for s in p.get("evidence_sentences", []))
            writer.writerow([
                d["doc_id"], fact_snippet,
                "; ".join(sorted(gold)), "; ".join(sorted(pred)),
                "; ".join(sorted(gold & pred)), "; ".join(sorted(gold - pred)), "; ".join(sorted(pred - gold)),
                "YES" if gold == pred else "NO",
                evidence_all[:500], pred_explanation_of(rec)[:500],
            ])
    print(f"Wrote predicted-vs-gold comparison to: {path}")


write_comparison_csv(test_docs, all_predictions, COMPARISON_PATH)
print("\nDONE.")

Torch: 2.14.0+cu130 | CUDA available: True
Device: cuda
STEP 3: Loading task1.jsonl + ipc_sections_clean.json, building splits
Loaded 575 official IPC sections from ipc_sections_clean.json
Total docs: 525 | Train: 378 | Val: 42 | Test: 105

STEP 4: Building the hybrid label space (classifier vs retrieval-only)
7 distinct gold sections seen in TRAIN docs.
-> 7 kept as CLASSIFIER classes (frequency >= 3).
-> remaining 568 of 575 sections are RETRIEVAL-ONLY.

STEP 5: Supervised Classification Branch (InLegalBERT -> BiLSTM -> label-wise Attention -> Linear & Softmax)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[classifier] epoch 1/8 -- avg loss: 1.9112
[classifier] epoch 2/8 -- avg loss: 1.6245
[classifier] epoch 3/8 -- avg loss: 1.3026
[classifier] epoch 4/8 -- avg loss: 1.0993
[classifier] epoch 5/8 -- avg loss: 0.9635
[classifier] epoch 6/8 -- avg loss: 0.8437
[classifier] epoch 7/8 -- avg loss: 0.7742
[classifier] epoch 8/8 -- avg loss: 0.7430
Classifier saved to ./classifier_out

STEP 6: IPC Retrieval Branch (BM25 + Cosine Similarity)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building BM25 index over the IPC Knowledge Base ...
Embedding the IPC Knowledge Base once for cosine-similarity retrieval ...
Catalog embedding matrix: (575, 768)

STEP 7: Calibrating per-class classifier thresholds on VAL
Calibrated 7 per-class thresholds: {'IPC 147': 0.12000000000000001, 'IPC 201': 0.06, 'IPC 302': 0.19999999999999998, 'IPC 376': 0.19999999999999998, 'IPC 420': 0.16, 'IPC 498A': 0.46, 'IPC 506': 0.1}

STEP 8: Fusion -> thresholded multi-label statute prediction

STEP 9: Evidence Sentence Retrieval for each predicted IPC

STEP 10: Loading Qwen (Qwen/Qwen2.5-1.5B-Instruct) for reasoning generation


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


STEP 11: Predicting on 105 held-out TEST documents
2002.INSC.274.txt -> pred=['IPC 376'] | gold=['IPC 376']
1999.INSC.378.txt -> pred=['IPC 201', 'IPC 376'] | gold=['IPC 201', 'IPC 302']
2012.INSC.512.txt -> pred=['IPC 201', 'IPC 302'] | gold=['IPC 498A']
1998.INSC.126.txt -> pred=['IPC 147', 'IPC 201', 'IPC 302'] | gold=['IPC 302']
2007.INSC.590.txt -> pred=['IPC 147', 'IPC 201', 'IPC 302'] | gold=['IPC 147']
2013.INSC.960.txt -> pred=['IPC 201', 'IPC 420', 'IPC 506'] | gold=['IPC 201']
2009.INSC.1130.txt -> pred=['IPC 376'] | gold=['IPC 376']
1999.INSC.175.txt -> pred=['IPC 147', 'IPC 302'] | gold=['IPC 302']
2003.INSC.597.txt -> pred=['IPC 147', 'IPC 302'] | gold=['IPC 302']
1998.INSC.474.txt -> pred=['IPC 302'] | gold=['IPC 302']
2015.INSC.345.txt -> pred=['IPC 147', 'IPC 201', 'IPC 302'] | gold=['IPC 302', 'IPC 506']
2011.INSC.316.txt -> pred=['IPC 147', 'IPC 302'] | gold=['IPC 147', 'IPC 302']
2016.INSC.429.txt -> pred=['IPC 201'] | gold=['IPC 201']
2007.INSC.291.txt -> pred=['I